# NammaFlow – Phase 3: Feature Engineering

## Objective

The objective of Phase 3 is to transform the cleaned and explored
Namma Yatri Bengaluru datasets into structured, model-ready features.

This phase builds features representing:

- Temporal demand patterns
- Search and booking behaviour
- Ride conversion performance
- Cancellation behaviour
- Ward-level funnel efficiency
- Driver availability and utilization
- Demand pressure relative to driver supply
- Operational performance across vehicle types

These engineered features will support the upcoming phases of:

- Demand–Supply Intelligence
- Supervised Machine Learning
- Unsupervised Learning
- Ward-Level Intelligence
- Recommendation Engine

Phase 1 completed data understanding and structural preprocessing.

Phase 2 completed exploratory analysis and identified major demand,
conversion, cancellation, funnel, and driver-supply patterns.

Phase 3 converts those findings into reusable analytical and
machine-learning features.

In [1]:
import os

repo_path = "/content/NammaFlow"

if os.path.exists(repo_path):
    print("NammaFlow repository already exists.")
else:
    !git clone https://github.com/Narasimhan15/NammaFlow.git

%cd /content/NammaFlow

!git status

Cloning into 'NammaFlow'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 40 (delta 12), reused 25 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 1.01 MiB | 14.42 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/NammaFlow
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


## 1. Environment Setup

Import the libraries required for feature engineering, data manipulation,
validation, and numerical analysis.

In [2]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Phase 3 environment ready.")

Phase 3 environment ready.


## 2. Load Processed Datasets

Before loading the processed datasets, verify the available files created during Phase 1.

In [3]:
processed_path = "data/processed"

print("Files available in data/processed:\n")

for file in sorted(os.listdir(processed_path)):
    print(file)

Files available in data/processed:

city_demand_bengaluru.csv
city_operations_bengaluru.csv
date_coverage.csv
driver_supply_clean.csv
ward_funnel_clean.csv


In [4]:
city_demand = pd.read_csv(
    f"{processed_path}/city_demand_bengaluru.csv"
)

city_operations = pd.read_csv(
    f"{processed_path}/city_operations_bengaluru.csv"
)

ward_funnel = pd.read_csv(
    f"{processed_path}/ward_funnel_clean.csv"
)

driver_supply = pd.read_csv(
    f"{processed_path}/driver_supply_clean.csv"
)

date_coverage = pd.read_csv(
    f"{processed_path}/date_coverage.csv"
)

print("Datasets loaded successfully.\n")

print("city_demand     :", city_demand.shape)
print("city_operations :", city_operations.shape)
print("ward_funnel     :", ward_funnel.shape)
print("driver_supply   :", driver_supply.shape)
print("date_coverage   :", date_coverage.shape)

Datasets loaded successfully.

city_demand     : (306, 24)
city_operations : (505, 24)
ward_funnel     : (2928, 29)
driver_supply   : (486, 6)
date_coverage   : (3, 3)


## 3. Data Type Preparation

Before engineering new features, the data types of each dataset are
inspected and validated.

Correct data types are especially important for:

- Date-based feature extraction
- Hour and time-period calculations
- Numerical aggregations
- Categorical feature handling
- Dataset merging and model preparation

No transformations are applied until the existing schema is verified.

In [5]:
datasets = {
    "city_demand": city_demand,
    "city_operations": city_operations,
    "ward_funnel": ward_funnel,
    "driver_supply": driver_supply,
    "date_coverage": date_coverage
}

for name, df in datasets.items():
    print("=" * 60)
    print(name.upper())
    print("=" * 60)
    print(df.dtypes)
    print()

CITY_DEMAND
date                         object
hour                          int64
trip_category                object
vehicle_type                 object
srch_rqst                     int64
srch_which_got_e              int64
srch_fr_q                     int64
srch_which_got_q              int64
booking                       int64
rides                         int64
done_ride                     int64
earning                       int64
cancel_ride                   int64
drvr_cancel                   int64
rider_cancel                  int64
srch_fr_q_variant           float64
srch_which_got_q_variant    float64
cnvr_rate                   float64
q_accept_rate               float64
city                         object
booking_type                 object
city_code                    object
bkng_cancel_rate            float64
is_complete_day                bool
dtype: object

CITY_OPERATIONS
date                         object
hour                          int64
trip_category        

The important findings are:
- date is currently object in city_demand, city_operations, and date_coverage → convert to datetime
- hour is already int64 → keep it
- ward_num is object → keep it as text, because values such as b_181, w_0, etc. are identifiers, not numbers
- trip_category, vehicle_type, booking_type, city → keep as categorical/text for now
- Counts are already integers
- Rates are already floats
- Boolean flags such as is_bike, is_cab, is_auto, active, is_complete_day are already correct

### 3.1 Create Feature Engineering Working Copies

To preserve the validated Phase 1 processed datasets, separate working
copies are created for feature engineering.

Only the date columns require datatype conversion at this stage.
Ward identifiers remain as text because they are categorical geographic IDs,
not numerical quantities.

In [6]:
# Create independent working copies
city_demand_fe = city_demand.copy()
city_operations_fe = city_operations.copy()
ward_funnel_fe = ward_funnel.copy()
driver_supply_fe = driver_supply.copy()
date_coverage_fe = date_coverage.copy()

# Convert date columns to datetime
city_demand_fe["date"] = pd.to_datetime(city_demand_fe["date"])
city_operations_fe["date"] = pd.to_datetime(city_operations_fe["date"])
date_coverage_fe["date"] = pd.to_datetime(date_coverage_fe["date"])

print("Datatype preparation completed.\n")

print("city_demand date      :", city_demand_fe["date"].dtype)
print("city_operations date  :", city_operations_fe["date"].dtype)
print("date_coverage date    :", date_coverage_fe["date"].dtype)

print("\nHour range:")
print(
    city_demand_fe["hour"].min(),
    "to",
    city_demand_fe["hour"].max()
)

print("\nUnique wards:")
print("Ward funnel :", ward_funnel_fe["ward_num"].nunique())
print("Driver supply:", driver_supply_fe["ward_num"].nunique())

Datatype preparation completed.

city_demand date      : datetime64[ns]
city_operations date  : datetime64[ns]
date_coverage date    : datetime64[ns]

Hour range:
0 to 23

Unique wards:
Ward funnel : 244
Driver supply: 244


## 4. Temporal Feature Engineering

Mobility demand changes significantly throughout the day.

The raw `hour` feature is useful for interpretation but does not represent
the cyclical relationship between hours. For example, hour 23 and hour 0
are numerically far apart even though they are adjacent in time.

Temporal features are therefore created to capture:

- Day of week
- Day name
- Operational time periods
- Cyclical hour behaviour

Month and year features are intentionally excluded because the available
dataset covers only a very short time period and those features would
provide little useful variation.

In [7]:
def add_temporal_features(df):
    df = df.copy()

    # Calendar features
    df["day_of_week"] = df["date"].dt.dayofweek
    df["day_name"] = df["date"].dt.day_name()

    # Operational time blocks
    df["time_period"] = pd.cut(
        df["hour"],
        bins=[-1, 5, 11, 16, 20, 23],
        labels=[
            "Overnight",
            "Morning",
            "Afternoon",
            "Evening",
            "Night"
        ]
    )

    # Cyclical encoding of hour
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    return df


city_demand_fe = add_temporal_features(city_demand_fe)
city_operations_fe = add_temporal_features(city_operations_fe)

print("Temporal features created successfully.")

Temporal features created successfully.


In [8]:
temporal_features = [
    "date",
    "hour",
    "day_of_week",
    "day_name",
    "time_period",
    "hour_sin",
    "hour_cos"
]

print("CITY DEMAND TEMPORAL FEATURES")
display(city_demand_fe[temporal_features].head(10))

print("\nUnique days:")
print(city_demand_fe["day_name"].value_counts())

print("\nTime-period distribution:")
print(city_demand_fe["time_period"].value_counts().sort_index())

print("\nMissing values in engineered temporal features:")
print(
    city_demand_fe[
        ["day_of_week", "day_name", "time_period", "hour_sin", "hour_cos"]
    ].isna().sum()
)

CITY DEMAND TEMPORAL FEATURES


,date,hour,day_of_week,day_name,time_period,hour_sin,hour_cos
0,2026-05-03,0,6,Sunday,Overnight,0.000000,1.000000
1,2026-05-03,0,6,Sunday,Overnight,0.000000,1.000000
2,2026-05-03,0,6,Sunday,Overnight,0.000000,1.000000
3,2026-05-03,1,6,Sunday,Overnight,0.258819,0.965926
4,2026-05-03,1,6,Sunday,Overnight,0.258819,0.965926
5,2026-05-03,1,6,Sunday,Overnight,0.258819,0.965926
6,2026-05-03,2,6,Sunday,Overnight,0.500000,0.866025
7,2026-05-03,2,6,Sunday,Overnight,0.500000,0.866025
8,2026-05-03,2,6,Sunday,Overnight,0.500000,0.866025
9,2026-05-03,3,6,Sunday,Overnight,0.707107,0.707107



Unique days:
day_name
Sunday      191
Saturday    115
Name: count, dtype: int64

Time-period distribution:
time_period
Overnight    78
Morning      87
Afternoon    71
Evening      41
Night        29
Name: count, dtype: int64

Missing values in engineered temporal features:
day_of_week    0
day_name       0
time_period    0
hour_sin       0
hour_cos       0
dtype: int64


## 5. City Demand Feature Engineering

The city demand dataset represents the overall search and booking funnel
(`vehicle_type = "All"`).

The objective of this section is to engineer features that quantify:

- Search progression through the funnel
- Quote-stage supply gaps
- Booking efficiency
- Ride completion efficiency
- Funnel-stage losses
- Overall search-to-completed-ride performance

Existing validated metrics such as `cnvr_rate`, `q_accept_rate`, and
`bkng_cancel_rate` are retained. New features complement these metrics
rather than replacing them.

In [9]:
def safe_divide(numerator, denominator):
    """
    Safely divide two pandas Series.
    Returns NaN where the denominator is zero.
    """
    return numerator.div(denominator.replace(0, np.nan))


# -----------------------------
# Funnel efficiency rates
# -----------------------------

city_demand_fe["srch_to_e_rate"] = safe_divide(
    city_demand_fe["srch_which_got_e"],
    city_demand_fe["srch_rqst"]
)

city_demand_fe["e_to_q_srch_rate"] = safe_divide(
    city_demand_fe["srch_fr_q"],
    city_demand_fe["srch_which_got_e"]
)

city_demand_fe["q_to_booking_rate"] = safe_divide(
    city_demand_fe["booking"],
    city_demand_fe["srch_which_got_q"]
)

city_demand_fe["booking_to_ride_rate"] = safe_divide(
    city_demand_fe["rides"],
    city_demand_fe["booking"]
)

city_demand_fe["ride_completion_rate"] = safe_divide(
    city_demand_fe["done_ride"],
    city_demand_fe["rides"]
)

city_demand_fe["overall_completion_rate"] = safe_divide(
    city_demand_fe["done_ride"],
    city_demand_fe["srch_rqst"]
)


# -----------------------------
# Funnel-stage loss features
# -----------------------------

city_demand_fe["search_to_e_gap"] = (
    city_demand_fe["srch_rqst"]
    - city_demand_fe["srch_which_got_e"]
)

city_demand_fe["e_to_q_gap"] = (
    city_demand_fe["srch_which_got_e"]
    - city_demand_fe["srch_fr_q"]
)

city_demand_fe["q_supply_gap"] = (
    city_demand_fe["srch_fr_q"]
    - city_demand_fe["srch_which_got_q"]
)

city_demand_fe["q_to_booking_gap"] = (
    city_demand_fe["srch_which_got_q"]
    - city_demand_fe["booking"]
)

city_demand_fe["booking_to_ride_gap"] = (
    city_demand_fe["booking"]
    - city_demand_fe["rides"]
)

city_demand_fe["ride_completion_gap"] = (
    city_demand_fe["rides"]
    - city_demand_fe["done_ride"]
)

city_demand_fe["search_to_booking_gap"] = (
    city_demand_fe["srch_rqst"]
    - city_demand_fe["booking"]
)

print("City demand features created successfully.")

City demand features created successfully.


In [10]:
demand_feature_cols = [
    "srch_rqst",
    "srch_to_e_rate",
    "e_to_q_srch_rate",
    "q_accept_rate",
    "q_to_booking_rate",
    "cnvr_rate",
    "booking_to_ride_rate",
    "ride_completion_rate",
    "overall_completion_rate",
    "search_to_e_gap",
    "e_to_q_gap",
    "q_supply_gap",
    "q_to_booking_gap",
    "booking_to_ride_gap",
    "ride_completion_gap",
    "search_to_booking_gap"
]

display(city_demand_fe[demand_feature_cols].head(10))

print("\nEngineered rate ranges:")

rate_cols = [
    "srch_to_e_rate",
    "e_to_q_srch_rate",
    "q_to_booking_rate",
    "booking_to_ride_rate",
    "ride_completion_rate",
    "overall_completion_rate"
]

for col in rate_cols:
    print(
        f"{col:25s}",
        "min =", round(city_demand_fe[col].min(), 4),
        "| max =", round(city_demand_fe[col].max(), 4)
    )

print("\nNegative funnel gaps:")

gap_cols = [
    "search_to_e_gap",
    "e_to_q_gap",
    "q_supply_gap",
    "q_to_booking_gap",
    "booking_to_ride_gap",
    "ride_completion_gap",
    "search_to_booking_gap"
]

for col in gap_cols:
    print(f"{col:25s}", (city_demand_fe[col] < 0).sum())

print("\nMissing values in new demand features:")
print(city_demand_fe[rate_cols + gap_cols].isna().sum())

,srch_rqst,srch_to_e_rate,e_to_q_srch_rate,q_accept_rate,q_to_booking_rate,cnvr_rate,booking_to_ride_rate,ride_completion_rate,overall_completion_rate,search_to_e_gap,e_to_q_gap,q_supply_gap,q_to_booking_gap,booking_to_ride_gap,ride_completion_gap,search_to_booking_gap
0,18,1.000000,1.000000,1.0000,0.0,0.0,NaN,NaN,0.0,0,0,0,18,0,0,18
1,5132,0.997272,0.738765,0.6321,0.0,0.0,NaN,NaN,0.0,14,1337,1391,2390,0,0,5132
2,11,1.000000,1.000000,1.0000,0.0,0.0,NaN,NaN,0.0,0,0,0,11,0,0,11
3,24,0.875000,0.857143,1.0000,0.0,0.0,NaN,NaN,0.0,3,3,0,18,0,0,24
4,4228,0.994560,0.809275,0.5190,0.0,0.0,NaN,NaN,0.0,23,802,1637,1766,0,0,4228
5,5,1.000000,1.000000,1.0000,0.0,0.0,NaN,NaN,0.0,0,0,0,5,0,0,5
6,22,1.000000,1.000000,1.0000,0.0,0.0,NaN,NaN,0.0,0,0,0,22,0,0,22
7,1350,0.994074,0.704173,0.7630,0.0,0.0,NaN,NaN,0.0,8,397,224,721,0,0,1350
8,9,1.000000,1.000000,1.0000,0.0,0.0,NaN,NaN,0.0,0,0,0,9,0,0,9
9,23,0.826087,1.000000,1.0000,0.0,0.0,NaN,NaN,0.0,4,0,0,19,0,0,23



Engineered rate ranges:
srch_to_e_rate            min = 0.6 | max = 1.0
e_to_q_srch_rate          min = 0.3333 | max = 1.0
q_to_booking_rate         min = 0.0 | max = 0.0
booking_to_ride_rate      min = nan | max = nan
ride_completion_rate      min = nan | max = nan
overall_completion_rate   min = 0.0 | max = 0.0

Negative funnel gaps:
search_to_e_gap           0
e_to_q_gap                0
q_supply_gap              0
q_to_booking_gap          0
booking_to_ride_gap       0
ride_completion_gap       0
search_to_booking_gap     0

Missing values in new demand features:
srch_to_e_rate               0
e_to_q_srch_rate             0
q_to_booking_rate            0
booking_to_ride_rate       306
ride_completion_rate       306
overall_completion_rate      0
search_to_e_gap              0
e_to_q_gap                   0
q_supply_gap                 0
q_to_booking_gap             0
booking_to_ride_gap          0
ride_completion_gap          0
search_to_booking_gap        0
dtype: int64


### 5.1 Structural Validation of City Demand Features

Validation confirmed that the city demand dataset is primarily a
search-side demand dataset.

Booking and ride fields are structurally zero in this table, causing
booking-based ratios to become zero or undefined.

Therefore:

- Search and quote-stage features are engineered from `city_demand`.
- Booking, cancellation, ride, completion, and earnings features are
  engineered separately from `city_operations`.
- The two analytical grains are intentionally not mixed.

This prevents structurally invalid features from entering later
machine-learning and decision-support stages.

**Removing the inappropriate features**

In [11]:
invalid_demand_features = [
    "q_to_booking_rate",
    "booking_to_ride_rate",
    "ride_completion_rate",
    "overall_completion_rate",
    "q_to_booking_gap",
    "booking_to_ride_gap",
    "ride_completion_gap",
    "search_to_booking_gap"
]

city_demand_fe.drop(
    columns=[
        col for col in invalid_demand_features
        if col in city_demand_fe.columns
    ],
    inplace=True
)

print("Structurally invalid booking/ride features removed from city_demand.")

Structurally invalid booking/ride features removed from city_demand.


**Demand-side features**

In [12]:
# Search → Q request rate
city_demand_fe["search_to_q_request_rate"] = safe_divide(
    city_demand_fe["srch_fr_q"],
    city_demand_fe["srch_rqst"]
)

# Search → successful Q response rate
city_demand_fe["search_to_q_received_rate"] = safe_divide(
    city_demand_fe["srch_which_got_q"],
    city_demand_fe["srch_rqst"]
)

# Proportion of Q requests that failed to receive supply
city_demand_fe["q_supply_gap_rate"] = safe_divide(
    city_demand_fe["q_supply_gap"],
    city_demand_fe["srch_fr_q"]
)

# Searches that failed to obtain a successful Q-stage response
city_demand_fe["search_to_q_received_gap"] = (
    city_demand_fe["srch_rqst"]
    - city_demand_fe["srch_which_got_q"]
)

print("Corrected city demand features created successfully.")

Corrected city demand features created successfully.


**Validating the correct features**

In [13]:
corrected_demand_features = [
    "srch_rqst",
    "srch_which_got_e",
    "srch_fr_q",
    "srch_which_got_q",
    "srch_to_e_rate",
    "e_to_q_srch_rate",
    "search_to_q_request_rate",
    "q_accept_rate",
    "search_to_q_received_rate",
    "q_supply_gap_rate",
    "search_to_e_gap",
    "e_to_q_gap",
    "q_supply_gap",
    "search_to_q_received_gap"
]

display(
    city_demand_fe[corrected_demand_features].head(10)
)

print("\nRate ranges:")

corrected_rate_cols = [
    "srch_to_e_rate",
    "e_to_q_srch_rate",
    "search_to_q_request_rate",
    "q_accept_rate",
    "search_to_q_received_rate",
    "q_supply_gap_rate"
]

for col in corrected_rate_cols:
    print(
        f"{col:28s}",
        "min =", round(city_demand_fe[col].min(), 4),
        "| max =", round(city_demand_fe[col].max(), 4)
    )

print("\nNegative gaps:")

corrected_gap_cols = [
    "search_to_e_gap",
    "e_to_q_gap",
    "q_supply_gap",
    "search_to_q_received_gap"
]

for col in corrected_gap_cols:
    print(
        f"{col:28s}",
        (city_demand_fe[col] < 0).sum()
    )

print("\nMissing values:")
print(
    city_demand_fe[
        corrected_rate_cols + corrected_gap_cols
    ].isna().sum()
)

,srch_rqst,srch_which_got_e,srch_fr_q,srch_which_got_q,srch_to_e_rate,e_to_q_srch_rate,search_to_q_request_rate,q_accept_rate,search_to_q_received_rate,q_supply_gap_rate,search_to_e_gap,e_to_q_gap,q_supply_gap,search_to_q_received_gap
0,18,18,18,18,1.000000,1.000000,1.000000,1.0000,1.000000,0.000000,0,0,0,0
1,5132,5118,3781,2390,0.997272,0.738765,0.736750,0.6321,0.465705,0.367892,14,1337,1391,2742
2,11,11,11,11,1.000000,1.000000,1.000000,1.0000,1.000000,0.000000,0,0,0,0
3,24,21,18,18,0.875000,0.857143,0.750000,1.0000,0.750000,0.000000,3,3,0,6
4,4228,4205,3403,1766,0.994560,0.809275,0.804872,0.5190,0.417692,0.481046,23,802,1637,2462
5,5,5,5,5,1.000000,1.000000,1.000000,1.0000,1.000000,0.000000,0,0,0,0
6,22,22,22,22,1.000000,1.000000,1.000000,1.0000,1.000000,0.000000,0,0,0,0
7,1350,1342,945,721,0.994074,0.704173,0.700000,0.7630,0.534074,0.237037,8,397,224,629
8,9,9,9,9,1.000000,1.000000,1.000000,1.0000,1.000000,0.000000,0,0,0,0
9,23,19,19,19,0.826087,1.000000,0.826087,1.0000,0.826087,0.000000,4,0,0,4



Rate ranges:
srch_to_e_rate               min = 0.6 | max = 1.0
e_to_q_srch_rate             min = 0.3333 | max = 1.0
search_to_q_request_rate     min = 0.3333 | max = 1.0
q_accept_rate                min = 0.1818 | max = 1.0
search_to_q_received_rate    min = 0.1 | max = 1.0
q_supply_gap_rate            min = 0.0 | max = 0.8182

Negative gaps:
search_to_e_gap              0
e_to_q_gap                   0
q_supply_gap                 0
search_to_q_received_gap     0

Missing values:
srch_to_e_rate               0
e_to_q_srch_rate             0
search_to_q_request_rate     0
q_accept_rate                0
search_to_q_received_rate    0
q_supply_gap_rate            0
search_to_e_gap              0
e_to_q_gap                   0
q_supply_gap                 0
search_to_q_received_gap     0
dtype: int64


**What we just did in Section 5**

Very simply:
Search → E → Q Request → Q Received
We converted raw counts into rates and gaps.

For example:
Searches = 5,132
Q requests = 3,781
Q received = 2,390

Then:
Search → Q request rate
= 3781 / 5132
≈ 73.7%

and:
Q supply gap
= 3781 - 2390
= 1,391

So instead of merely knowing that there were 3,781 requests, NammaFlow can understand that 1,391 requests failed at the supply-response stage.
That distinction will become very important later.

## 6. City Operations Feature Engineering

The city operations dataset contains vehicle-level operational activity
for Auto and Cab services.

Unlike `city_demand`, this dataset is used for booking, ride,
cancellation, completion, and earnings-related features.

The objective is to engineer measures describing:

- Booking-to-ride efficiency
- Ride completion performance
- Booking completion performance
- Cancellation intensity
- Driver vs rider cancellation contribution
- Revenue efficiency
- Operational losses between funnel stages

These features represent supply-side execution and service quality rather
than overall search demand.

In [14]:
# ---------------------------------
# Operational conversion features
# ---------------------------------

city_operations_fe["booking_to_ride_rate"] = safe_divide(
    city_operations_fe["rides"],
    city_operations_fe["booking"]
)

city_operations_fe["ride_completion_rate"] = safe_divide(
    city_operations_fe["done_ride"],
    city_operations_fe["rides"]
)

city_operations_fe["booking_completion_rate"] = safe_divide(
    city_operations_fe["done_ride"],
    city_operations_fe["booking"]
)


# ---------------------------------
# Cancellation features
# ---------------------------------

city_operations_fe["cancel_per_booking"] = safe_divide(
    city_operations_fe["cancel_ride"],
    city_operations_fe["booking"]
)

city_operations_fe["driver_cancel_share"] = safe_divide(
    city_operations_fe["drvr_cancel"],
    city_operations_fe["cancel_ride"]
)

city_operations_fe["rider_cancel_share"] = safe_divide(
    city_operations_fe["rider_cancel"],
    city_operations_fe["cancel_ride"]
)


# ---------------------------------
# Revenue efficiency features
# ---------------------------------

city_operations_fe["earning_per_ride"] = safe_divide(
    city_operations_fe["earning"],
    city_operations_fe["rides"]
)

city_operations_fe["earning_per_completed_ride"] = safe_divide(
    city_operations_fe["earning"],
    city_operations_fe["done_ride"]
)


# ---------------------------------
# Operational loss features
# ---------------------------------

city_operations_fe["booking_to_ride_gap"] = (
    city_operations_fe["booking"]
    - city_operations_fe["rides"]
)

city_operations_fe["ride_completion_gap"] = (
    city_operations_fe["rides"]
    - city_operations_fe["done_ride"]
)

city_operations_fe["booking_to_completion_gap"] = (
    city_operations_fe["booking"]
    - city_operations_fe["done_ride"]
)

print("City operations features created successfully.")

City operations features created successfully.


**Validating**

In [15]:
operations_feature_cols = [
    "vehicle_type",
    "booking",
    "rides",
    "done_ride",
    "cancel_ride",
    "drvr_cancel",
    "rider_cancel",
    "earning",
    "booking_to_ride_rate",
    "ride_completion_rate",
    "booking_completion_rate",
    "cancel_per_booking",
    "driver_cancel_share",
    "rider_cancel_share",
    "earning_per_ride",
    "earning_per_completed_ride",
    "booking_to_ride_gap",
    "ride_completion_gap",
    "booking_to_completion_gap"
]

display(
    city_operations_fe[operations_feature_cols].head(10)
)

print("\nOperational rate ranges:")

operation_rate_cols = [
    "booking_to_ride_rate",
    "ride_completion_rate",
    "booking_completion_rate",
    "cancel_per_booking"
]

for col in operation_rate_cols:
    print(
        f"{col:30s}",
        "min =", round(city_operations_fe[col].min(), 4),
        "| max =", round(city_operations_fe[col].max(), 4)
    )

print("\nNegative operational gaps:")

operation_gap_cols = [
    "booking_to_ride_gap",
    "ride_completion_gap",
    "booking_to_completion_gap"
]

for col in operation_gap_cols:
    print(
        f"{col:30s}",
        (city_operations_fe[col] < 0).sum()
    )

print("\nZero denominators:")

print(
    "booking = 0 :",
    (city_operations_fe["booking"] == 0).sum()
)

print(
    "rides = 0   :",
    (city_operations_fe["rides"] == 0).sum()
)

print(
    "done_ride = 0:",
    (city_operations_fe["done_ride"] == 0).sum()
)

print(
    "cancel_ride = 0:",
    (city_operations_fe["cancel_ride"] == 0).sum()
)

print("\nMissing values created by safe division:")

engineered_operation_cols = [
    "booking_to_ride_rate",
    "ride_completion_rate",
    "booking_completion_rate",
    "cancel_per_booking",
    "driver_cancel_share",
    "rider_cancel_share",
    "earning_per_ride",
    "earning_per_completed_ride"
]

print(
    city_operations_fe[
        engineered_operation_cols
    ].isna().sum()
)

,vehicle_type,booking,rides,done_ride,cancel_ride,drvr_cancel,rider_cancel,earning,booking_to_ride_rate,ride_completion_rate,booking_completion_rate,cancel_per_booking,driver_cancel_share,rider_cancel_share,earning_per_ride,earning_per_completed_ride,booking_to_ride_gap,ride_completion_gap,booking_to_completion_gap
0,Cab,3,2,1,2,0,2,3800,0.666667,0.500000,0.333333,0.666667,0.000000,1.000000,1900.000000,3800.000000,1,1,2
1,Auto,1164,1164,663,501,264,237,126608,1.000000,0.569588,0.569588,0.430412,0.526946,0.473054,108.769759,190.962293,0,501,501
2,Cab,936,720,383,524,134,378,139631,0.769231,0.531944,0.409188,0.559829,0.255725,0.721374,193.931944,364.571802,216,337,553
3,Auto,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
4,Cab,1,1,0,1,0,1,0,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,NaN,0,1,1
5,Cab,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
6,Auto,785,785,400,385,198,187,83625,1.000000,0.509554,0.509554,0.490446,0.514286,0.485714,106.528662,209.062500,0,385,385
7,Cab,660,523,240,407,126,271,103101,0.792424,0.458891,0.363636,0.616667,0.309582,0.665848,197.133843,429.587500,137,283,420
8,Auto,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
9,Cab,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0



Operational rate ranges:
booking_to_ride_rate           min = 0.0 | max = 1.25
ride_completion_rate           min = 0.0 | max = 1.0
booking_completion_rate        min = 0.0 | max = 1.0
cancel_per_booking             min = 0.0 | max = 1.0

Negative operational gaps:
booking_to_ride_gap            3
ride_completion_gap            0
booking_to_completion_gap      0

Zero denominators:
booking = 0 : 81
rides = 0   : 89
done_ride = 0: 130
cancel_ride = 0: 104

Missing values created by safe division:
booking_to_ride_rate           81
ride_completion_rate           89
booking_completion_rate        81
cancel_per_booking             81
driver_cancel_share           104
rider_cancel_share            104
earning_per_ride               89
earning_per_completed_ride    130
dtype: int64


### 6.1 Validate Booking-to-Ride Anomalies

Validation identified a small number of operational rows where the number
of rides exceeds the number of bookings.

These rows are inspected before any correction is applied.

No clipping, deletion, or imputation is performed automatically because
the apparent inconsistency may reflect the underlying source-data
definition or aggregation behaviour.

In [16]:
booking_ride_anomalies = city_operations_fe[
    city_operations_fe["rides"] > city_operations_fe["booking"]
].copy()

print("Rows where rides > booking:",
      len(booking_ride_anomalies))

display(
    booking_ride_anomalies[
        [
            "date",
            "hour",
            "trip_category",
            "vehicle_type",
            "booking_type",
            "booking",
            "rides",
            "done_ride",
            "cancel_ride",
            "drvr_cancel",
            "rider_cancel",
            "booking_to_ride_rate",
            "booking_to_ride_gap"
        ]
    ]
)

Rows where rides > booking: 3


,date,hour,trip_category,vehicle_type,booking_type,booking,rides,done_ride,cancel_ride,drvr_cancel,rider_cancel,booking_to_ride_rate,booking_to_ride_gap
58,2026-05-03,11,Rental,Auto,Normal,6,7,5,1,1,0,1.166667,-1
284,2026-05-10,9,InterCity,Cab,Normal,26,27,4,22,7,13,1.038462,-1
490,2026-05-10,11,InterCity,Cab,Purple,4,5,0,4,0,3,1.250000,-1


In [17]:
print("Counts for anomalous rows:\n")

print(
    booking_ride_anomalies[
        [
            "booking",
            "rides",
            "done_ride",
            "cancel_ride"
        ]
    ].to_string(index=False)
)

print("\nMaximum excess rides over bookings:")

print(
    (booking_ride_anomalies["rides"]
     - booking_ride_anomalies["booking"]).max()
)

Counts for anomalous rows:

 booking  rides  done_ride  cancel_ride
       6      7          5            1
      26     27          4           22
       4      5          0            4

Maximum excess rides over bookings:
1


What Section 6 is doing

Think of the data as two different questions.

city_demand tells us:
**Are customers finding supply?**

city_operations tells us:
**Once operational activity happens,**
**how efficiently does the service perform?**

So features such as:
**booking_completion_rate**
**= completed rides / bookings**

tell us service execution quality.
And:
**driver_cancel_share**
**= driver cancellations / total cancellations**

helps answer:
**Is cancellation mainly being caused by drivers or riders?**

While:
**earning_per_completed_ride**
**= earnings / completed rides**

describes revenue efficiency.
Later, NammaFlow can combine signals conceptually like:
High demand
+ poor quote supply
+ high driver cancellation
+ low completion
→ operational intervention needed

That's the business reason for this feature engineering.


### 6.2 Operational Consistency Flags

Three rows contain slightly more rides than bookings, with an excess of
exactly one ride in each case.

Because the dataset is aggregated into hourly operational buckets, this
may reflect temporal carry-over between booking and ride activity.

The records are therefore preserved rather than clipped or removed.

An explicit consistency flag is created so later models and diagnostics
can identify these observations.

In [18]:
city_operations_fe["ride_booking_mismatch"] = (
    city_operations_fe["rides"]
    > city_operations_fe["booking"]
).astype(int)

city_operations_fe["rides_minus_bookings"] = (
    city_operations_fe["rides"]
    - city_operations_fe["booking"]
)

print(
    "Ride-booking mismatch rows:",
    city_operations_fe["ride_booking_mismatch"].sum()
)

print(
    "Maximum rides above bookings:",
    city_operations_fe["rides_minus_bookings"].max()
)

Ride-booking mismatch rows: 3
Maximum rides above bookings: 1


In [19]:
city_operations_fe["other_cancel"] = (
    city_operations_fe["cancel_ride"]
    - city_operations_fe["drvr_cancel"]
    - city_operations_fe["rider_cancel"]
)

city_operations_fe["other_cancel_share"] = safe_divide(
    city_operations_fe["other_cancel"],
    city_operations_fe["cancel_ride"]
)

print(
    "Rows with unclassified cancellations:",
    (city_operations_fe["other_cancel"] > 0).sum()
)

print(
    "Negative unclassified cancellation counts:",
    (city_operations_fe["other_cancel"] < 0).sum()
)

print(
    "Maximum unclassified cancellations:",
    city_operations_fe["other_cancel"].max()
)

Rows with unclassified cancellations: 153
Negative unclassified cancellation counts: 0
Maximum unclassified cancellations: 76


## 7. Ward-Level Funnel Feature Engineering

The ward funnel dataset provides geographic-level demand and operational
performance across Bengaluru wards.

Phase 2 identified large differences in conversion performance between
wards. Phase 3 converts those observations into reusable features.

The engineered ward-level features measure:

- Search funnel leakage
- Quote-stage supply failure
- Booking efficiency
- Booking-to-completion performance
- Cancellation behaviour
- Revenue efficiency
- End-to-end search completion performance

Existing validated funnel rates are retained and complemented with
additional gap and efficiency features.

In [20]:
# ---------------------------------
# Additional ward funnel rates
# ---------------------------------

ward_funnel_fe["search_to_q_request_rate"] = safe_divide(
    ward_funnel_fe["srch_fr_q"],
    ward_funnel_fe["srch_fr_e"]
)

ward_funnel_fe["search_to_q_received_rate"] = safe_divide(
    ward_funnel_fe["srch_which_got_q"],
    ward_funnel_fe["srch_fr_e"]
)

ward_funnel_fe["booking_completion_rate"] = safe_divide(
    ward_funnel_fe["done_ride"],
    ward_funnel_fe["booking"]
)

ward_funnel_fe["search_completion_rate"] = safe_divide(
    ward_funnel_fe["done_ride"],
    ward_funnel_fe["srch_fr_e"]
)


# ---------------------------------
# Cancellation features
# ---------------------------------

ward_funnel_fe["cancel_per_booking"] = safe_divide(
    ward_funnel_fe["cancel_ride"],
    ward_funnel_fe["booking"]
)

ward_funnel_fe["driver_cancel_share"] = safe_divide(
    ward_funnel_fe["drvr_cancel"],
    ward_funnel_fe["cancel_ride"]
)

ward_funnel_fe["rider_cancel_share"] = safe_divide(
    ward_funnel_fe["rider_cancel"],
    ward_funnel_fe["cancel_ride"]
)


# ---------------------------------
# Revenue efficiency
# ---------------------------------

ward_funnel_fe["earning_per_completed_ride"] = safe_divide(
    ward_funnel_fe["earning"],
    ward_funnel_fe["done_ride"]
)


# ---------------------------------
# Funnel leakage / gap features
# ---------------------------------

ward_funnel_fe["search_to_e_gap"] = (
    ward_funnel_fe["srch_fr_e"]
    - ward_funnel_fe["srch_which_got_e"]
)

ward_funnel_fe["e_to_q_gap"] = (
    ward_funnel_fe["srch_which_got_e"]
    - ward_funnel_fe["srch_fr_q"]
)

ward_funnel_fe["q_supply_gap"] = (
    ward_funnel_fe["srch_fr_q"]
    - ward_funnel_fe["srch_which_got_q"]
)

ward_funnel_fe["q_to_booking_gap"] = (
    ward_funnel_fe["srch_which_got_q"]
    - ward_funnel_fe["booking"]
)

ward_funnel_fe["booking_to_completion_gap"] = (
    ward_funnel_fe["booking"]
    - ward_funnel_fe["done_ride"]
)

ward_funnel_fe["search_to_completion_gap"] = (
    ward_funnel_fe["srch_fr_e"]
    - ward_funnel_fe["done_ride"]
)

print("Ward-level funnel features created successfully.")

Ward-level funnel features created successfully.


In Phase 2, we discovered wards such as b_181 with high demand but poor conversion. Now we need to convert those raw ward funnel numbers into machine-readable features.

In [21]:
ward_feature_cols = [
    "ward_num",
    "trip_category",
    "vehicle_type",
    "srch_fr_e",
    "srch_to_e_rate",
    "e_to_q_srch_rate",
    "q_accept_rate",
    "q_to_bkng_rate",
    "cnvr_rate",
    "search_to_q_request_rate",
    "search_to_q_received_rate",
    "booking_completion_rate",
    "search_completion_rate",
    "cancel_per_booking",
    "driver_cancel_share",
    "rider_cancel_share",
    "earning_per_completed_ride",
    "search_to_e_gap",
    "e_to_q_gap",
    "q_supply_gap",
    "q_to_booking_gap",
    "booking_to_completion_gap",
    "search_to_completion_gap"
]

display(
    ward_funnel_fe[ward_feature_cols].head(10)
)

print("\nNew ward rate ranges:")

new_ward_rate_cols = [
    "search_to_q_request_rate",
    "search_to_q_received_rate",
    "booking_completion_rate",
    "search_completion_rate",
    "cancel_per_booking"
]

for col in new_ward_rate_cols:
    print(
        f"{col:30s}",
        "min =", round(ward_funnel_fe[col].min(), 4),
        "| max =", round(ward_funnel_fe[col].max(), 4)
    )

print("\nNegative ward funnel gaps:")

ward_gap_cols = [
    "search_to_e_gap",
    "e_to_q_gap",
    "q_supply_gap",
    "q_to_booking_gap",
    "booking_to_completion_gap",
    "search_to_completion_gap"
]

for col in ward_gap_cols:
    print(
        f"{col:30s}",
        (ward_funnel_fe[col] < 0).sum()
    )

print("\nMissing values in new ward features:")

print(
    ward_funnel_fe[
        new_ward_rate_cols
        + ["driver_cancel_share",
           "rider_cancel_share",
           "earning_per_completed_ride"]
        + ward_gap_cols
    ].isna().sum()
)

,ward_num,trip_category,vehicle_type,srch_fr_e,srch_to_e_rate,e_to_q_srch_rate,q_accept_rate,q_to_bkng_rate,cnvr_rate,search_to_q_request_rate,search_to_q_received_rate,booking_completion_rate,search_completion_rate,cancel_per_booking,driver_cancel_share,rider_cancel_share,earning_per_completed_ride,search_to_e_gap,e_to_q_gap,q_supply_gap,q_to_booking_gap,booking_to_completion_gap,search_to_completion_gap
0,b_1,InterCity,All,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0
1,b_1,InterCity,Auto,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0
2,b_1,InterCity,Bike,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0
3,b_1,InterCity,Cab,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0
4,b_1,OneWay,All,70,1.0,0.5286,0.5135,0.0,0.0,0.528571,0.271429,NaN,0.0,NaN,NaN,NaN,NaN,0,33,18,19,0,70
5,b_1,OneWay,Auto,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.700000,NaN,0.200000,0.5,0.5,181.428571,0,0,0,-10,3,-7
6,b_1,OneWay,Bike,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0
7,b_1,OneWay,Cab,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.555556,NaN,0.444444,0.5,0.5,247.400000,0,0,0,-9,4,-5
8,b_1,Rental,All,2,1.0,1.0000,1.0000,0.0,0.0,1.000000,1.000000,NaN,0.0,NaN,NaN,NaN,NaN,0,0,0,2,0,2
9,b_1,Rental,Auto,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0



New ward rate ranges:
search_to_q_request_rate       min = 0.0 | max = 1.0
search_to_q_received_rate      min = 0.0 | max = 1.0
booking_completion_rate        min = 0.0 | max = 1.0
search_completion_rate         min = 0.0 | max = 0.0
cancel_per_booking             min = 0.0 | max = 1.0

Negative ward funnel gaps:
search_to_e_gap                0
e_to_q_gap                     0
q_supply_gap                   0
q_to_booking_gap               469
booking_to_completion_gap      0
search_to_completion_gap       432

Missing values in new ward features:
search_to_q_request_rate      2614
search_to_q_received_rate     2614
booking_completion_rate       2459
search_completion_rate        2614
cancel_per_booking            2459
driver_cancel_share           2525
rider_cancel_share            2525
earning_per_completed_ride    2496
search_to_e_gap                  0
e_to_q_gap                       0
q_supply_gap                     0
q_to_booking_gap                 0
booking_to_completion_ga

What we're doing here

Suppose one ward has:
1,000 searches,
800 Q requests,
400 Q responses,
350 bookings,
150 completed rides.

Raw numbers alone tell us activity volume.

Feature engineering converts them into business signals:
Q supply gap
=> 800 - 400
= 400

Meaning **400 customer requests failed at the quote/supply stage.**
And:
Booking completion rate
= 150 / 350
≈ 42.9%

Meaning fewer than half the bookings ultimately became completed rides.
And:
Search completion rate
= 150 / 1000
= 15%

So only 15% of original search demand ultimately became completed trips.

That's the difference between EDA and feature engineering:

EDA helps us discover the problem.
Feature engineering converts the discovered problem into numerical signals that ML models and the NammaFlow recommendation engine can use.

### 7.1 Structural Separation of Ward Demand and Operations

Validation revealed that the ward dataset contains two different
analytical row roles.

- `vehicle_type = "All"` contains ward-level search and demand funnel data.
- `vehicle_type = Auto / Bike / Cab` contains vehicle-level operational data.

Therefore, search-side and operational metrics must not be combined
directly at the raw row level.

The ward dataset is split into:

1. `ward_demand_fe` – search and quote-stage demand features.
2. `ward_operations_fe` – bookings, rides, cancellations, and earnings.

These tables will later be aggregated to the common ward level before
being combined.

In [22]:
# Identify the analytical role of each row
ward_funnel_fe["row_role"] = np.where(
    ward_funnel_fe["vehicle_type"].eq("All"),
    "Demand",
    "Operations"
)

# Split the dataset by analytical role
ward_demand_fe = ward_funnel_fe[
    ward_funnel_fe["row_role"] == "Demand"
].copy()

ward_operations_fe = ward_funnel_fe[
    ward_funnel_fe["row_role"] == "Operations"
].copy()

print("Ward demand rows     :", ward_demand_fe.shape)
print("Ward operations rows :", ward_operations_fe.shape)

print("\nDemand vehicle types:")
print(ward_demand_fe["vehicle_type"].value_counts())

print("\nOperational vehicle types:")
print(ward_operations_fe["vehicle_type"].value_counts())

print("\nUnique wards:")
print("Demand     :", ward_demand_fe["ward_num"].nunique())
print("Operations :", ward_operations_fe["ward_num"].nunique())

Ward demand rows     : (732, 44)
Ward operations rows : (2196, 44)

Demand vehicle types:
vehicle_type
All    732
Name: count, dtype: int64

Operational vehicle types:
vehicle_type
Auto    732
Bike    732
Cab     732
Name: count, dtype: int64

Unique wards:
Demand     : 244
Operations : 244


**Demand-side ward features**

In [23]:
# ---------------------------------
# Valid ward demand features
# ---------------------------------

ward_demand_fe["search_to_q_request_rate"] = safe_divide(
    ward_demand_fe["srch_fr_q"],
    ward_demand_fe["srch_fr_e"]
)

ward_demand_fe["search_to_q_received_rate"] = safe_divide(
    ward_demand_fe["srch_which_got_q"],
    ward_demand_fe["srch_fr_e"]
)

ward_demand_fe["q_supply_gap"] = (
    ward_demand_fe["srch_fr_q"]
    - ward_demand_fe["srch_which_got_q"]
)

ward_demand_fe["q_supply_gap_rate"] = safe_divide(
    ward_demand_fe["q_supply_gap"],
    ward_demand_fe["srch_fr_q"]
)

ward_demand_fe["search_to_e_gap"] = (
    ward_demand_fe["srch_fr_e"]
    - ward_demand_fe["srch_which_got_e"]
)

ward_demand_fe["e_to_q_gap"] = (
    ward_demand_fe["srch_which_got_e"]
    - ward_demand_fe["srch_fr_q"]
)

ward_demand_fe["search_to_q_received_gap"] = (
    ward_demand_fe["srch_fr_e"]
    - ward_demand_fe["srch_which_got_q"]
)

print("Valid ward demand features created.")

Valid ward demand features created.


**Operational ward features**

In [24]:
# ---------------------------------
# Valid ward operational features
# ---------------------------------

ward_operations_fe["booking_to_ride_rate"] = safe_divide(
    ward_operations_fe["rides"],
    ward_operations_fe["booking"]
)

ward_operations_fe["ride_completion_rate"] = safe_divide(
    ward_operations_fe["done_ride"],
    ward_operations_fe["rides"]
)

ward_operations_fe["booking_completion_rate"] = safe_divide(
    ward_operations_fe["done_ride"],
    ward_operations_fe["booking"]
)

ward_operations_fe["cancel_per_booking"] = safe_divide(
    ward_operations_fe["cancel_ride"],
    ward_operations_fe["booking"]
)

ward_operations_fe["driver_cancel_share"] = safe_divide(
    ward_operations_fe["drvr_cancel"],
    ward_operations_fe["cancel_ride"]
)

ward_operations_fe["rider_cancel_share"] = safe_divide(
    ward_operations_fe["rider_cancel"],
    ward_operations_fe["cancel_ride"]
)

ward_operations_fe["earning_per_ride"] = safe_divide(
    ward_operations_fe["earning"],
    ward_operations_fe["rides"]
)

ward_operations_fe["earning_per_completed_ride"] = safe_divide(
    ward_operations_fe["earning"],
    ward_operations_fe["done_ride"]
)

ward_operations_fe["booking_to_ride_gap"] = (
    ward_operations_fe["booking"]
    - ward_operations_fe["rides"]
)

ward_operations_fe["ride_completion_gap"] = (
    ward_operations_fe["rides"]
    - ward_operations_fe["done_ride"]
)

ward_operations_fe["booking_to_completion_gap"] = (
    ward_operations_fe["booking"]
    - ward_operations_fe["done_ride"]
)

print("Valid ward operational features created.")

Valid ward operational features created.


**Why this correction matters**


We are essentially building two puzzle pieces:

WARD DEMAND,

Search → E → Q request → Q received

and separately:

WARD OPERATIONS

Booking → Ride → Completed ride
              ↓
          Cancellation
              ↓
            Earnings

Later, after aggregating both to one row per ward, we can safely combine them:

Ward b_181
│
├── Demand pressure
├── Quote supply failure
├── Bookings
├── Completed rides
├── Cancellations
├── Earnings
└── Driver availability

That combined ward table is what will eventually make NammaFlow powerful.


This confirms the architecture is now correct: **244 wards × 3 trip categories = 732 demand rows, while 244 wards × 3 trip categories × 3 operational vehicle types = 2,196 operational rows.**

### 7.2 Validation of Corrected Ward Feature Tables

The separated ward demand and ward operations tables are validated
independently.

Demand-side validation focuses on search and quote-stage consistency.

Operational validation focuses on booking, ride, completion,
cancellation, and revenue consistency.

Structural missing values caused by zero denominators are preserved as
undefined values rather than being artificially replaced.

**Ward demand validation**

In [25]:
ward_demand_rate_cols = [
    "srch_to_e_rate",
    "e_to_q_srch_rate",
    "search_to_q_request_rate",
    "q_accept_rate",
    "search_to_q_received_rate",
    "q_supply_gap_rate"
]

ward_demand_gap_cols = [
    "search_to_e_gap",
    "e_to_q_gap",
    "q_supply_gap",
    "search_to_q_received_gap"
]

print("WARD DEMAND VALIDATION")
print("=" * 50)

print("\nRate ranges:")
for col in ward_demand_rate_cols:
    print(
        f"{col:30s}",
        "min =", round(ward_demand_fe[col].min(), 4),
        "| max =", round(ward_demand_fe[col].max(), 4)
    )

print("\nNegative demand gaps:")
for col in ward_demand_gap_cols:
    print(
        f"{col:30s}",
        (ward_demand_fe[col] < 0).sum()
    )

print("\nMissing values:")
print(
    ward_demand_fe[
        ward_demand_rate_cols + ward_demand_gap_cols
    ].isna().sum()
)

print("\nZero search denominators:")
print(
    "srch_fr_e = 0 :",
    (ward_demand_fe["srch_fr_e"] == 0).sum()
)

print(
    "srch_fr_q = 0 :",
    (ward_demand_fe["srch_fr_q"] == 0).sum()
)

WARD DEMAND VALIDATION

Rate ranges:
srch_to_e_rate                 min = 0.3333 | max = 1.0
e_to_q_srch_rate               min = 0.0 | max = 1.0
search_to_q_request_rate       min = 0.0 | max = 1.0
q_accept_rate                  min = 0.0 | max = 1.0
search_to_q_received_rate      min = 0.0 | max = 1.0
q_supply_gap_rate              min = 0.0 | max = 1.0

Negative demand gaps:
search_to_e_gap                0
e_to_q_gap                     0
q_supply_gap                   0
search_to_q_received_gap       0

Missing values:
srch_to_e_rate               418
e_to_q_srch_rate             418
search_to_q_request_rate     418
q_accept_rate                419
search_to_q_received_rate    418
q_supply_gap_rate            419
search_to_e_gap                0
e_to_q_gap                     0
q_supply_gap                   0
search_to_q_received_gap       0
dtype: int64

Zero search denominators:
srch_fr_e = 0 : 418
srch_fr_q = 0 : 419


**Operational consistency features**

In [26]:
# Ride-booking consistency
ward_operations_fe["ride_booking_mismatch"] = (
    ward_operations_fe["rides"]
    > ward_operations_fe["booking"]
).astype(int)

ward_operations_fe["rides_minus_bookings"] = (
    ward_operations_fe["rides"]
    - ward_operations_fe["booking"]
)

# Unclassified cancellation component
ward_operations_fe["other_cancel"] = (
    ward_operations_fe["cancel_ride"]
    - ward_operations_fe["drvr_cancel"]
    - ward_operations_fe["rider_cancel"]
)

ward_operations_fe["other_cancel_share"] = safe_divide(
    ward_operations_fe["other_cancel"],
    ward_operations_fe["cancel_ride"]
)

print("Ward operational consistency features created.")

Ward operational consistency features created.


**Validating the operational side**

In [27]:
ward_operation_rate_cols = [
    "booking_to_ride_rate",
    "ride_completion_rate",
    "booking_completion_rate",
    "cancel_per_booking"
]

ward_operation_gap_cols = [
    "booking_to_ride_gap",
    "ride_completion_gap",
    "booking_to_completion_gap"
]

print("WARD OPERATIONS VALIDATION")
print("=" * 50)

print("\nRate ranges:")
for col in ward_operation_rate_cols:
    print(
        f"{col:30s}",
        "min =", round(ward_operations_fe[col].min(), 4),
        "| max =", round(ward_operations_fe[col].max(), 4)
    )

print("\nNegative operational gaps:")
for col in ward_operation_gap_cols:
    print(
        f"{col:30s}",
        (ward_operations_fe[col] < 0).sum()
    )

print("\nRide-booking mismatches:")
print(
    ward_operations_fe["ride_booking_mismatch"].sum()
)

print("\nMaximum rides minus bookings:")
print(
    ward_operations_fe["rides_minus_bookings"].max()
)

print("\nCancellation consistency:")
print(
    "Rows with other cancellations:",
    (ward_operations_fe["other_cancel"] > 0).sum()
)

print(
    "Negative other cancellations:",
    (ward_operations_fe["other_cancel"] < 0).sum()
)

print(
    "Maximum other cancellations:",
    ward_operations_fe["other_cancel"].max()
)

print("\nZero denominators:")
print("booking = 0     :", (ward_operations_fe["booking"] == 0).sum())
print("rides = 0       :", (ward_operations_fe["rides"] == 0).sum())
print("cancel_ride = 0 :", (ward_operations_fe["cancel_ride"] == 0).sum())

WARD OPERATIONS VALIDATION

Rate ranges:
booking_to_ride_rate           min = 0.5 | max = 1.3333
ride_completion_rate           min = 0.0 | max = 1.0
booking_completion_rate        min = 0.0 | max = 1.0
cancel_per_booking             min = 0.0 | max = 1.0

Negative operational gaps:
booking_to_ride_gap            1
ride_completion_gap            0
booking_to_completion_gap      0

Ride-booking mismatches:
1

Maximum rides minus bookings:
1

Cancellation consistency:
Rows with other cancellations: 13
Negative other cancellations: 0
Maximum other cancellations: 2

Zero denominators:
booking = 0     : 1727
rides = 0       : 1727
cancel_ride = 0 : 1793


### 7.3 Inspect Ward Ride–Booking Mismatch

A single operational ward record contains one more ride than booking.

The record is inspected before any correction is applied. Because the
difference is only one observation, the row is preserved unless there is
clear evidence of a data-quality error.

In [28]:
ward_ride_booking_anomaly = ward_operations_fe[
    ward_operations_fe["rides"] > ward_operations_fe["booking"]
].copy()

print(
    "Ward ride-booking anomaly rows:",
    len(ward_ride_booking_anomaly)
)

display(
    ward_ride_booking_anomaly[
        [
            "ward_num",
            "trip_category",
            "vehicle_type",
            "booking",
            "rides",
            "done_ride",
            "cancel_ride",
            "drvr_cancel",
            "rider_cancel",
            "booking_to_ride_rate",
            "rides_minus_bookings"
        ]
    ]
)

Ward ride-booking anomaly rows: 1


,ward_num,trip_category,vehicle_type,booking,rides,done_ride,cancel_ride,drvr_cancel,rider_cancel,booking_to_ride_rate,rides_minus_bookings
2915,b_99,Rental,Cab,3,4,0,3,0,2,1.333333,1


The difference is only one ride, and we already saw the same tiny carry-over pattern in the city operations data, we should keep this record unchanged and retain our ride_booking_mismatch flag. We should not “correct” genuine source data just to force every ratio below-1.

## 8. Driver Supply Feature Engineering

Driver availability is a critical component of NammaFlow because high
customer demand does not automatically translate into successful rides.

This section engineers features describing:

- Active driver supply
- Drivers currently on rides
- Drivers available but not on rides
- Driver utilization
- Idle / available supply share
- Driver-count consistency
- Vehicle-level supply coverage across wards

Before engineering these features, the internal relationship between
driver counts is validated.

**Diagnostic**

In [29]:
print("DRIVER SUPPLY STRUCTURE")
print("=" * 50)

print("Shape:", driver_supply_fe.shape)

print("\nVehicle types:")
print(driver_supply_fe["vehicle_type"].value_counts())

print("\nActive flag:")
print(driver_supply_fe["active"].value_counts(dropna=False))

print("\nUnique wards:")
print(driver_supply_fe["ward_num"].nunique())

print("\nDriver count summary:")
display(
    driver_supply_fe[
        [
            "active_drvr",
            "drvr_notonride",
            "drvr_onride"
        ]
    ].describe()
)

# Check whether active drivers equal on-ride + not-on-ride drivers
driver_supply_fe["driver_count_gap"] = (
    driver_supply_fe["active_drvr"]
    - driver_supply_fe["drvr_notonride"]
    - driver_supply_fe["drvr_onride"]
)

print("\nDriver count consistency:")

print(
    "Rows where active_drvr != notonride + onride:",
    (driver_supply_fe["driver_count_gap"] != 0).sum()
)

print(
    "Minimum driver_count_gap:",
    driver_supply_fe["driver_count_gap"].min()
)

print(
    "Maximum driver_count_gap:",
    driver_supply_fe["driver_count_gap"].max()
)

print("\nZero active-driver rows:")
print(
    (driver_supply_fe["active_drvr"] == 0).sum()
)

DRIVER SUPPLY STRUCTURE
Shape: (486, 6)

Vehicle types:
vehicle_type
Auto    244
Cab     242
Name: count, dtype: int64

Active flag:
active
True    486
Name: count, dtype: int64

Unique wards:
244

Driver count summary:


,active_drvr,drvr_notonride,drvr_onride
count,486.000000,486.000000,486.000000
mean,100.742798,87.041152,13.701646
std,287.103799,272.441596,20.671497
min,1.000000,0.000000,0.000000
25%,26.250000,21.000000,2.000000
50%,53.000000,45.500000,7.000000
75%,109.000000,92.000000,16.750000
max,5109.000000,4921.000000,196.000000



Driver count consistency:
Rows where active_drvr != notonride + onride: 0
Minimum driver_count_gap: 0
Maximum driver_count_gap: 0

Zero active-driver rows:
0


### 8.1 Driver Utilization and Availability Features

Driver counts are internally consistent across all records:

`active_drvr = drvr_notonride + drvr_onride`

This allows utilization and not-on-ride supply ratios to be calculated
reliably.

The following features are created:

- Driver utilization rate
- Not-on-ride driver share
- On-ride to not-on-ride ratio
- Log-transformed active driver supply

The log-transformed feature reduces the effect of extremely large driver
counts while preserving the ordering of supply levels.

In [30]:
# ---------------------------------
# Driver utilization features
# ---------------------------------

driver_supply_fe["driver_utilization_rate"] = safe_divide(
    driver_supply_fe["drvr_onride"],
    driver_supply_fe["active_drvr"]
)

driver_supply_fe["notonride_driver_share"] = safe_divide(
    driver_supply_fe["drvr_notonride"],
    driver_supply_fe["active_drvr"]
)

# Ratio between occupied and currently not-on-ride drivers
driver_supply_fe["onride_to_notonride_ratio"] = safe_divide(
    driver_supply_fe["drvr_onride"],
    driver_supply_fe["drvr_notonride"]
)

# Log-transformed driver supply for later ML use
driver_supply_fe["log_active_drvr"] = np.log1p(
    driver_supply_fe["active_drvr"]
)

print("Driver supply features created successfully.")

Driver supply features created successfully.


In [31]:
print("DRIVER SUPPLY FEATURE VALIDATION")
print("=" * 55)

driver_rate_cols = [
    "driver_utilization_rate",
    "notonride_driver_share",
    "onride_to_notonride_ratio"
]

print("\nFeature ranges:")

for col in driver_rate_cols:
    print(
        f"{col:30s}",
        "min =", round(driver_supply_fe[col].min(), 4),
        "| max =", round(driver_supply_fe[col].max(), 4)
    )

# Check complementary shares
driver_supply_fe["driver_share_sum"] = (
    driver_supply_fe["driver_utilization_rate"]
    + driver_supply_fe["notonride_driver_share"]
)

print("\nUtilization + not-on-ride share:")

print(
    "Minimum:",
    round(driver_supply_fe["driver_share_sum"].min(), 6)
)

print(
    "Maximum:",
    round(driver_supply_fe["driver_share_sum"].max(), 6)
)

print("\nMissing engineered features:")

print(
    driver_supply_fe[
        [
            "driver_utilization_rate",
            "notonride_driver_share",
            "onride_to_notonride_ratio",
            "log_active_drvr"
        ]
    ].isna().sum()
)

print("\nRows with zero not-on-ride drivers:")

print(
    (driver_supply_fe["drvr_notonride"] == 0).sum()
)

DRIVER SUPPLY FEATURE VALIDATION

Feature ranges:
driver_utilization_rate        min = 0.0 | max = 1.0
notonride_driver_share         min = 0.0 | max = 1.0
onride_to_notonride_ratio      min = 0.0 | max = 1.4

Utilization + not-on-ride share:
Minimum: 1.0
Maximum: 1.0

Missing engineered features:
driver_utilization_rate      0
notonride_driver_share       0
onride_to_notonride_ratio    2
log_active_drvr              0
dtype: int64

Rows with zero not-on-ride drivers:
2


**Confirm the two incomplete supply wards**

In [32]:
auto_wards = set(
    driver_supply_fe.loc[
        driver_supply_fe["vehicle_type"] == "Auto",
        "ward_num"
    ]
)

cab_wards = set(
    driver_supply_fe.loc[
        driver_supply_fe["vehicle_type"] == "Cab",
        "ward_num"
    ]
)

all_supply_wards = auto_wards | cab_wards

incomplete_supply_wards = sorted(
    ward
    for ward in all_supply_wards
    if not (
        ward in auto_wards
        and ward in cab_wards
    )
)

print("Auto-covered wards :", len(auto_wards))
print("Cab-covered wards  :", len(cab_wards))
print("Complete Auto + Cab:", len(auto_wards & cab_wards))

print("\nIncomplete supply wards:")
print(incomplete_supply_wards)

Auto-covered wards : 244
Cab-covered wards  : 242
Complete Auto + Cab: 242

Incomplete supply wards:
['b_156', 'b_77']


**What these features mean**

Suppose a ward has:
active_drvr     = 100
drvr_onride     = 80
drvr_notonride  = 20

Then:
driver_utilization_rate
= 80 / 100
= 80%

and:
notonride_driver_share
= 20 / 100
= 20%

Now NammaFlow can distinguish:

High demand + 90% driver utilization
→ drivers are already heavily occupied
→ likely real supply shortage

from:
High demand + 30% utilization
→ many drivers are not on rides
→ may be a matching / positioning / allocation problem

That is a much stronger business feature than simply saying “this ward has 100 drivers.”

## 9. Build the Ward Intelligence Master Table

The final Phase 3 ward-level feature table requires a common analytical
grain.

Demand, operations, and driver-supply datasets currently have different
row structures. Therefore, each source is first aggregated independently
to one row per ward.

Raw counts are aggregated first, and ratio-based features are recalculated
from the aggregated totals rather than averaging row-level percentages.

This prevents weighting errors and preserves the correct business meaning
of each feature.

### 9.1 Ward Demand Summary

Ward-level search and quote-stage demand are aggregated across trip
categories to create one demand record per ward.

Rates are recalculated from aggregated counts rather than averaged from
row-level rates.

In [33]:
# ============================================
# 9.1 Aggregate demand features to ward level
# ============================================

ward_demand_summary = (
    ward_demand_fe
    .groupby("ward_num", as_index=False)
    .agg(
        ward_searches=("srch_fr_e", "sum"),
        ward_searches_got_e=("srch_which_got_e", "sum"),
        ward_q_requests=("srch_fr_q", "sum"),
        ward_q_received=("srch_which_got_q", "sum")
    )
)

# --------------------------------------------
# Recalculate rates from aggregated totals
# --------------------------------------------

ward_demand_summary["ward_search_to_e_rate"] = safe_divide(
    ward_demand_summary["ward_searches_got_e"],
    ward_demand_summary["ward_searches"]
)

ward_demand_summary["ward_e_to_q_request_rate"] = safe_divide(
    ward_demand_summary["ward_q_requests"],
    ward_demand_summary["ward_searches_got_e"]
)

ward_demand_summary["ward_q_accept_rate"] = safe_divide(
    ward_demand_summary["ward_q_received"],
    ward_demand_summary["ward_q_requests"]
)

ward_demand_summary["ward_search_to_q_request_rate"] = safe_divide(
    ward_demand_summary["ward_q_requests"],
    ward_demand_summary["ward_searches"]
)

ward_demand_summary["ward_search_to_q_received_rate"] = safe_divide(
    ward_demand_summary["ward_q_received"],
    ward_demand_summary["ward_searches"]
)

# --------------------------------------------
# Demand-side gap features
# --------------------------------------------

ward_demand_summary["ward_q_supply_gap"] = (
    ward_demand_summary["ward_q_requests"]
    - ward_demand_summary["ward_q_received"]
)

ward_demand_summary["ward_q_supply_gap_rate"] = safe_divide(
    ward_demand_summary["ward_q_supply_gap"],
    ward_demand_summary["ward_q_requests"]
)

ward_demand_summary["ward_search_to_q_received_gap"] = (
    ward_demand_summary["ward_searches"]
    - ward_demand_summary["ward_q_received"]
)

print("Ward demand summary created successfully.\n")

print("Shape:", ward_demand_summary.shape)
print(
    "Unique wards:",
    ward_demand_summary["ward_num"].nunique()
)

print(
    "Duplicate wards:",
    ward_demand_summary["ward_num"].duplicated().sum()
)

Ward demand summary created successfully.

Shape: (244, 13)
Unique wards: 244
Duplicate wards: 0


In [34]:
display(ward_demand_summary.head())

print("\nNegative demand gaps:")

print(
    "ward_q_supply_gap:",
    (ward_demand_summary["ward_q_supply_gap"] < 0).sum()
)

print(
    "ward_search_to_q_received_gap:",
    (ward_demand_summary["ward_search_to_q_received_gap"] < 0).sum()
)

print("\nMissing values:")

print(
    ward_demand_summary.isna().sum()
)

,ward_num,ward_searches,ward_searches_got_e,ward_q_requests,ward_q_received,ward_search_to_e_rate,ward_e_to_q_request_rate,ward_q_accept_rate,ward_search_to_q_request_rate,ward_search_to_q_received_rate,ward_q_supply_gap,ward_q_supply_gap_rate,ward_search_to_q_received_gap
0,b_1,72,72,39,21,1.0,0.541667,0.538462,0.541667,0.291667,18,0.461538,51
1,b_10,188,188,139,65,1.0,0.739362,0.467626,0.739362,0.345745,74,0.532374,123
2,b_100,159,159,129,56,1.0,0.811321,0.434109,0.811321,0.352201,73,0.565891,103
3,b_101,43,43,29,19,1.0,0.674419,0.655172,0.674419,0.441860,10,0.344828,24
4,b_102,127,127,110,52,1.0,0.866142,0.472727,0.866142,0.409449,58,0.527273,75



Negative demand gaps:
ward_q_supply_gap: 0
ward_search_to_q_received_gap: 0

Missing values:
ward_num                          0
ward_searches                     0
ward_searches_got_e               0
ward_q_requests                   0
ward_q_received                   0
ward_search_to_e_rate             0
ward_e_to_q_request_rate          0
ward_q_accept_rate                1
ward_search_to_q_request_rate     0
ward_search_to_q_received_rate    0
ward_q_supply_gap                 0
ward_q_supply_gap_rate            1
ward_search_to_q_received_gap     0
dtype: int64


### 9.2 Ward Operations Summary

Vehicle-level operational records are aggregated to one row per ward.

Bookings, rides, completed rides, cancellations, and earnings are summed
first. Operational rates are then recalculated from the aggregated totals.

This preserves volume weighting and avoids averaging percentages across
vehicle types or trip categories.

In [35]:
# ==============================================
# 9.2 Aggregate operations to one row per ward
# ==============================================

ward_operations_summary = (
    ward_operations_fe
    .groupby("ward_num", as_index=False)
    .agg(
        ward_bookings=("booking", "sum"),
        ward_rides=("rides", "sum"),
        ward_completed_rides=("done_ride", "sum"),
        ward_cancellations=("cancel_ride", "sum"),
        ward_driver_cancellations=("drvr_cancel", "sum"),
        ward_rider_cancellations=("rider_cancel", "sum"),
        ward_earnings=("earning", "sum")
    )
)

# ----------------------------------------------
# Operational efficiency
# ----------------------------------------------

ward_operations_summary["ward_booking_to_ride_rate"] = safe_divide(
    ward_operations_summary["ward_rides"],
    ward_operations_summary["ward_bookings"]
)

ward_operations_summary["ward_ride_completion_rate"] = safe_divide(
    ward_operations_summary["ward_completed_rides"],
    ward_operations_summary["ward_rides"]
)

ward_operations_summary["ward_booking_completion_rate"] = safe_divide(
    ward_operations_summary["ward_completed_rides"],
    ward_operations_summary["ward_bookings"]
)

# ----------------------------------------------
# Cancellation features
# ----------------------------------------------

ward_operations_summary["ward_cancel_per_booking"] = safe_divide(
    ward_operations_summary["ward_cancellations"],
    ward_operations_summary["ward_bookings"]
)

ward_operations_summary["ward_other_cancellations"] = (
    ward_operations_summary["ward_cancellations"]
    - ward_operations_summary["ward_driver_cancellations"]
    - ward_operations_summary["ward_rider_cancellations"]
)

ward_operations_summary["ward_driver_cancel_share"] = safe_divide(
    ward_operations_summary["ward_driver_cancellations"],
    ward_operations_summary["ward_cancellations"]
)

ward_operations_summary["ward_rider_cancel_share"] = safe_divide(
    ward_operations_summary["ward_rider_cancellations"],
    ward_operations_summary["ward_cancellations"]
)

ward_operations_summary["ward_other_cancel_share"] = safe_divide(
    ward_operations_summary["ward_other_cancellations"],
    ward_operations_summary["ward_cancellations"]
)

# ----------------------------------------------
# Revenue efficiency
# ----------------------------------------------

ward_operations_summary["ward_earning_per_ride"] = safe_divide(
    ward_operations_summary["ward_earnings"],
    ward_operations_summary["ward_rides"]
)

ward_operations_summary["ward_earning_per_completed_ride"] = safe_divide(
    ward_operations_summary["ward_earnings"],
    ward_operations_summary["ward_completed_rides"]
)

# ----------------------------------------------
# Operational gap features
# ----------------------------------------------

ward_operations_summary["ward_booking_to_ride_gap"] = (
    ward_operations_summary["ward_bookings"]
    - ward_operations_summary["ward_rides"]
)

ward_operations_summary["ward_ride_completion_gap"] = (
    ward_operations_summary["ward_rides"]
    - ward_operations_summary["ward_completed_rides"]
)

ward_operations_summary["ward_booking_to_completion_gap"] = (
    ward_operations_summary["ward_bookings"]
    - ward_operations_summary["ward_completed_rides"]
)

# Aggregated mismatch flag
ward_operations_summary["ward_ride_booking_mismatch"] = (
    ward_operations_summary["ward_rides"]
    > ward_operations_summary["ward_bookings"]
).astype(int)

print("Ward operations summary created successfully.\n")

print("Shape:", ward_operations_summary.shape)

print(
    "Unique wards:",
    ward_operations_summary["ward_num"].nunique()
)

print(
    "Duplicate wards:",
    ward_operations_summary["ward_num"].duplicated().sum()
)

Ward operations summary created successfully.

Shape: (244, 22)
Unique wards: 244
Duplicate wards: 0


In [36]:
display(ward_operations_summary.head())

print("\nOperational rate ranges:")

operation_summary_rate_cols = [
    "ward_booking_to_ride_rate",
    "ward_ride_completion_rate",
    "ward_booking_completion_rate",
    "ward_cancel_per_booking"
]

for col in operation_summary_rate_cols:
    print(
        f"{col:35s}",
        "min =", round(ward_operations_summary[col].min(), 4),
        "| max =", round(ward_operations_summary[col].max(), 4)
    )

print("\nNegative operational gaps:")

operation_summary_gap_cols = [
    "ward_booking_to_ride_gap",
    "ward_ride_completion_gap",
    "ward_booking_to_completion_gap"
]

for col in operation_summary_gap_cols:
    print(
        f"{col:35s}",
        (ward_operations_summary[col] < 0).sum()
    )

print("\nRide-booking mismatch wards:")

print(
    ward_operations_summary[
        "ward_ride_booking_mismatch"
    ].sum()
)

print("\nCancellation consistency:")

print(
    "Negative other cancellations:",
    (ward_operations_summary["ward_other_cancellations"] < 0).sum()
)

print(
    "Wards with other cancellations:",
    (ward_operations_summary["ward_other_cancellations"] > 0).sum()
)

print("\nMissing values:")

print(
    ward_operations_summary.isna().sum()
)

,ward_num,ward_bookings,ward_rides,ward_completed_rides,ward_cancellations,ward_driver_cancellations,ward_rider_cancellations,ward_earnings,ward_booking_to_ride_rate,ward_ride_completion_rate,ward_booking_completion_rate,ward_cancel_per_booking,ward_other_cancellations,ward_driver_cancel_share,ward_rider_cancel_share,ward_other_cancel_share,ward_earning_per_ride,ward_earning_per_completed_ride,ward_booking_to_ride_gap,ward_ride_completion_gap,ward_booking_to_completion_gap,ward_ride_booking_mismatch
0,b_1,19,19,12,6,3,3,2507,1.0,0.631579,0.631579,0.315789,0,0.500000,0.500000,0.0,131.947368,208.916667,0,7,7,0
1,b_10,65,65,32,27,15,12,8413,1.0,0.492308,0.492308,0.415385,0,0.555556,0.444444,0.0,129.430769,262.906250,0,33,33,0
2,b_100,56,56,37,13,3,10,6783,1.0,0.660714,0.660714,0.232143,0,0.230769,0.769231,0.0,121.125000,183.324324,0,19,19,0
3,b_101,19,19,5,9,4,5,1118,1.0,0.263158,0.263158,0.473684,0,0.444444,0.555556,0.0,58.842105,223.600000,0,14,14,0
4,b_102,50,50,23,24,10,14,5165,1.0,0.460000,0.460000,0.480000,0,0.416667,0.583333,0.0,103.300000,224.565217,0,27,27,0



Operational rate ranges:
ward_booking_to_ride_rate           min = 0.7165 | max = 1.0087
ward_ride_completion_rate           min = 0.0 | max = 1.0
ward_booking_completion_rate        min = 0.0 | max = 1.0
ward_cancel_per_booking             min = 0.0 | max = 1.0

Negative operational gaps:
ward_booking_to_ride_gap            1
ward_ride_completion_gap            0
ward_booking_to_completion_gap      0

Ride-booking mismatch wards:
1

Cancellation consistency:
Negative other cancellations: 0
Wards with other cancellations: 11

Missing values:
ward_num                            0
ward_bookings                       0
ward_rides                          0
ward_completed_rides                0
ward_cancellations                  0
ward_driver_cancellations           0
ward_rider_cancellations            0
ward_earnings                       0
ward_booking_to_ride_rate           3
ward_ride_completion_rate           3
ward_booking_completion_rate        3
ward_cancel_per_booking          

Why this is different from what we did earlier

Earlier, ward_operations_fe still had rows such as:

b_99 + Rental + Cab
b_99 + OneWay + Auto
b_99 + InterCity + Bike
...

Now we're creating:

b_99

→ total bookings
→ total rides
→ total completed rides
→ total cancellations
→ total earnings
→ recalculated operational efficiency

So after this step, one row represents the total operational condition of an entire ward.

### 9.3 Ward Driver-Supply Summary

Auto and Cab driver-supply records are aggregated to one row per ward.

The resulting features represent:

- Total active Auto + Cab drivers
- Drivers currently on rides
- Drivers currently not on rides
- Ward-level driver utilization
- Available driver share
- Auto/Cab supply coverage

Because Cab supply is missing for two wards, explicit coverage flags are
retained instead of treating missing Cab records as zero supply.

In [37]:
# ==============================================
# 9.3 Aggregate Auto + Cab supply to ward level
# ==============================================

ward_supply_summary = (
    driver_supply_fe
    .groupby("ward_num", as_index=False)
    .agg(
        supply_vehicle_coverage=("vehicle_type", "nunique"),
        active_drvr_auto_cab=("active_drvr", "sum"),
        notonride_drvr_auto_cab=("drvr_notonride", "sum"),
        onride_drvr_auto_cab=("drvr_onride", "sum")
    )
)

# ----------------------------------------------
# Explicit vehicle coverage flags
# ----------------------------------------------

ward_supply_summary["has_auto_supply"] = (
    ward_supply_summary["ward_num"].isin(auto_wards)
).astype(int)

ward_supply_summary["has_cab_supply"] = (
    ward_supply_summary["ward_num"].isin(cab_wards)
).astype(int)

ward_supply_summary["complete_auto_cab_supply"] = (
    (ward_supply_summary["has_auto_supply"] == 1)
    & (ward_supply_summary["has_cab_supply"] == 1)
).astype(int)

# ----------------------------------------------
# Ward-level driver utilization
# ----------------------------------------------

ward_supply_summary["ward_driver_utilization_rate"] = safe_divide(
    ward_supply_summary["onride_drvr_auto_cab"],
    ward_supply_summary["active_drvr_auto_cab"]
)

ward_supply_summary["ward_notonride_driver_share"] = safe_divide(
    ward_supply_summary["notonride_drvr_auto_cab"],
    ward_supply_summary["active_drvr_auto_cab"]
)

ward_supply_summary["ward_log_active_drivers"] = np.log1p(
    ward_supply_summary["active_drvr_auto_cab"]
)

print("Ward supply summary created successfully.\n")

print("Shape:", ward_supply_summary.shape)

print(
    "Unique wards:",
    ward_supply_summary["ward_num"].nunique()
)

print(
    "Duplicate wards:",
    ward_supply_summary["ward_num"].duplicated().sum()
)

print(
    "Complete Auto + Cab wards:",
    ward_supply_summary["complete_auto_cab_supply"].sum()
)

Ward supply summary created successfully.

Shape: (244, 11)
Unique wards: 244
Duplicate wards: 0
Complete Auto + Cab wards: 242


In [38]:
display(ward_supply_summary.head())

print("\nDriver count consistency:")

ward_supply_summary["ward_driver_count_gap"] = (
    ward_supply_summary["active_drvr_auto_cab"]
    - ward_supply_summary["notonride_drvr_auto_cab"]
    - ward_supply_summary["onride_drvr_auto_cab"]
)

print(
    "Non-zero driver count gaps:",
    (ward_supply_summary["ward_driver_count_gap"] != 0).sum()
)

print("\nDriver share consistency:")

ward_supply_summary["ward_driver_share_sum"] = (
    ward_supply_summary["ward_driver_utilization_rate"]
    + ward_supply_summary["ward_notonride_driver_share"]
)

print(
    "Minimum utilization + available share:",
    round(ward_supply_summary["ward_driver_share_sum"].min(), 6)
)

print(
    "Maximum utilization + available share:",
    round(ward_supply_summary["ward_driver_share_sum"].max(), 6)
)

print("\nIncomplete Auto/Cab coverage:")

display(
    ward_supply_summary.loc[
        ward_supply_summary["complete_auto_cab_supply"] == 0,
        [
            "ward_num",
            "has_auto_supply",
            "has_cab_supply",
            "supply_vehicle_coverage",
            "active_drvr_auto_cab",
            "notonride_drvr_auto_cab",
            "onride_drvr_auto_cab"
        ]
    ]
)

print("\nMissing values:")

print(
    ward_supply_summary.isna().sum()
)

,ward_num,supply_vehicle_coverage,active_drvr_auto_cab,notonride_drvr_auto_cab,onride_drvr_auto_cab,has_auto_supply,has_cab_supply,complete_auto_cab_supply,ward_driver_utilization_rate,ward_notonride_driver_share,ward_log_active_drivers
0,b_1,2,336,313,23,1,1,1,0.068452,0.931548,5.820083
1,b_10,2,491,399,92,1,1,1,0.187373,0.812627,6.198479
2,b_100,2,297,217,80,1,1,1,0.269360,0.730640,5.697093
3,b_101,2,86,60,26,1,1,1,0.302326,0.697674,4.465908
4,b_102,2,87,64,23,1,1,1,0.264368,0.735632,4.477337



Driver count consistency:
Non-zero driver count gaps: 0

Driver share consistency:
Minimum utilization + available share: 1.0
Maximum utilization + available share: 1.0

Incomplete Auto/Cab coverage:


,ward_num,has_auto_supply,has_cab_supply,supply_vehicle_coverage,active_drvr_auto_cab,notonride_drvr_auto_cab,onride_drvr_auto_cab
63,b_156,1,0,1,16,14,2
218,b_77,1,0,1,1,1,0



Missing values:
ward_num                        0
supply_vehicle_coverage         0
active_drvr_auto_cab            0
notonride_drvr_auto_cab         0
onride_drvr_auto_cab            0
has_auto_supply                 0
has_cab_supply                  0
complete_auto_cab_supply        0
ward_driver_utilization_rate    0
ward_notonride_driver_share     0
ward_log_active_drivers         0
ward_driver_count_gap           0
ward_driver_share_sum           0
dtype: int64


What Section 9.3 is doing

Earlier we might have had:
b_181 + Auto
b_181 + Cab

Now we create:
b_181
│
├── total Auto + Cab active drivers
├── drivers currently on rides
├── drivers currently not on rides
├── driver utilization %
├── available driver %
└── supply coverage status

After this, all three tables will finally have exactly one row per ward:
ward_demand_summary
        244 rows

ward_operations_summary
        244 rows

ward_supply_summary
        244 rows

### 9.4 Ward Alignment and Controlled Merge

Before combining demand, operations, and driver-supply features, the ward
keys are validated.

Each source must contain:

- Exactly 244 wards
- No duplicated ward IDs
- The same set of ward IDs

Only after these conditions are confirmed is a one-to-one merge performed.

Using explicit one-to-one merge validation prevents accidental row
multiplication and protects the integrity of the final feature table.

In [39]:
# ==============================================
# 9.4 Validate ward alignment before merging
# ==============================================

demand_wards = set(
    ward_demand_summary["ward_num"]
)

operations_wards = set(
    ward_operations_summary["ward_num"]
)

supply_wards = set(
    ward_supply_summary["ward_num"]
)

print("WARD ALIGNMENT VALIDATION")
print("=" * 55)

print(
    "Demand wards     :",
    len(demand_wards)
)

print(
    "Operations wards :",
    len(operations_wards)
)

print(
    "Supply wards     :",
    len(supply_wards)
)

print("\nWard-set equality:")

print(
    "Demand = Operations:",
    demand_wards == operations_wards
)

print(
    "Demand = Supply:",
    demand_wards == supply_wards
)

print(
    "Operations = Supply:",
    operations_wards == supply_wards
)

print("\nMissing from operations:")
print(
    sorted(demand_wards - operations_wards)
)

print("\nMissing from supply:")
print(
    sorted(demand_wards - supply_wards)
)

WARD ALIGNMENT VALIDATION
Demand wards     : 244
Operations wards : 244
Supply wards     : 244

Ward-set equality:
Demand = Operations: True
Demand = Supply: True
Operations = Supply: True

Missing from operations:
[]

Missing from supply:
[]


### 9.5 NammaFlow Ward Intelligence Master Table

The validated demand, operations, and driver-supply summaries are merged
using `ward_num` as the geographic key.

A strict one-to-one merge is enforced so that every row in the resulting
table represents exactly one Bengaluru ward.

The resulting master table combines:

- Demand pressure
- Search funnel performance
- Quote-stage supply gaps
- Booking and ride execution
- Cancellation behaviour
- Earnings efficiency
- Driver availability
- Driver utilization
- Supply coverage quality

This table becomes the primary ward-level feature dataset for later
NammaFlow intelligence, machine-learning, and recommendation stages.

In [40]:
# ==============================================
# Build NammaFlow Ward Intelligence Master Table
# ==============================================

ward_intelligence = (
    ward_demand_summary
    .merge(
        ward_operations_summary,
        on="ward_num",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        ward_supply_summary,
        on="ward_num",
        how="inner",
        validate="one_to_one"
    )
)

print("Ward Intelligence Master Table created successfully.\n")

print("Shape:", ward_intelligence.shape)

print(
    "Unique wards:",
    ward_intelligence["ward_num"].nunique()
)

print(
    "Duplicate wards:",
    ward_intelligence["ward_num"].duplicated().sum()
)

Ward Intelligence Master Table created successfully.

Shape: (244, 46)
Unique wards: 244
Duplicate wards: 0


In [41]:
print("MASTER TABLE VALIDATION")
print("=" * 55)

print("\nRows and columns:")
print(ward_intelligence.shape)

print("\nDuplicate wards:")
print(
    ward_intelligence["ward_num"].duplicated().sum()
)

print("\nMissing values by column:")

master_missing = (
    ward_intelligence
    .isna()
    .sum()
)

print(
    master_missing[
        master_missing > 0
    ].sort_values(ascending=False)
)

print("\nSupply coverage:")

print(
    ward_intelligence[
        "complete_auto_cab_supply"
    ].value_counts()
)

print("\nIncomplete supply wards:")

display(
    ward_intelligence.loc[
        ward_intelligence["complete_auto_cab_supply"] == 0,
        [
            "ward_num",
            "ward_searches",
            "ward_q_requests",
            "ward_q_received",
            "ward_bookings",
            "ward_completed_rides",
            "active_drvr_auto_cab",
            "notonride_drvr_auto_cab",
            "onride_drvr_auto_cab",
            "has_auto_supply",
            "has_cab_supply",
            "complete_auto_cab_supply"
        ]
    ]
)

MASTER TABLE VALIDATION

Rows and columns:
(244, 46)

Duplicate wards:
0

Missing values by column:
ward_other_cancel_share            19
ward_rider_cancel_share            19
ward_driver_cancel_share           19
ward_earning_per_completed_ride     7
ward_cancel_per_booking             3
ward_booking_to_ride_rate           3
ward_booking_completion_rate        3
ward_ride_completion_rate           3
ward_earning_per_ride               3
ward_q_supply_gap_rate              1
ward_q_accept_rate                  1
dtype: int64

Supply coverage:
complete_auto_cab_supply
1    242
0      2
Name: count, dtype: int64

Incomplete supply wards:


,ward_num,ward_searches,ward_q_requests,ward_q_received,ward_bookings,ward_completed_rides,active_drvr_auto_cab,notonride_drvr_auto_cab,onride_drvr_auto_cab,has_auto_supply,has_cab_supply,complete_auto_cab_supply
63,b_156,4,3,1,1,1,16,14,2,1,0,0
218,b_77,9,8,4,4,2,1,1,0,1,0,0


## 10. Cross-Domain Demand–Supply Feature Engineering

The Ward Intelligence Master Table makes it possible to create interaction
features across previously separate business domains.

These features connect customer demand, booking performance, ride
completion, and driver supply.

The objectives are to measure:

- Search-to-booking conversion
- Search-to-completed-ride conversion
- Quote-to-booking conversion
- End-to-end demand leakage
- Demand pressure per active driver
- Demand pressure per currently available driver
- Quote-stage supply failure relative to driver availability

Driver-pressure features are calculated only for wards with complete
Auto + Cab supply coverage. Wards with incomplete supply coverage are
preserved but their supply-pressure ratios remain undefined to avoid
misleading comparisons.

**End-to-End Funnel features**

In [42]:
# ==============================================
# 10.1 Cross-domain end-to-end funnel features
# ==============================================

ward_intelligence["ward_search_to_booking_rate"] = safe_divide(
    ward_intelligence["ward_bookings"],
    ward_intelligence["ward_searches"]
)

ward_intelligence["ward_search_to_completion_rate"] = safe_divide(
    ward_intelligence["ward_completed_rides"],
    ward_intelligence["ward_searches"]
)

ward_intelligence["ward_q_to_booking_rate"] = safe_divide(
    ward_intelligence["ward_bookings"],
    ward_intelligence["ward_q_received"]
)

# End-to-end leakage counts
ward_intelligence["ward_q_to_booking_gap"] = (
    ward_intelligence["ward_q_received"]
    - ward_intelligence["ward_bookings"]
)

ward_intelligence["ward_search_to_booking_gap"] = (
    ward_intelligence["ward_searches"]
    - ward_intelligence["ward_bookings"]
)

ward_intelligence["ward_search_to_completion_gap"] = (
    ward_intelligence["ward_searches"]
    - ward_intelligence["ward_completed_rides"]
)

print("End-to-end ward funnel features created.")

End-to-end ward funnel features created.


**Validation**

In [43]:
print("END-TO-END FUNNEL VALIDATION")
print("=" * 55)

cross_domain_rate_cols = [
    "ward_search_to_booking_rate",
    "ward_search_to_completion_rate",
    "ward_q_to_booking_rate"
]

print("\nRate ranges:")

for col in cross_domain_rate_cols:
    print(
        f"{col:35s}",
        "min =", round(ward_intelligence[col].min(), 4),
        "| max =", round(ward_intelligence[col].max(), 4)
    )

print("\nNegative end-to-end gaps:")

for col in [
    "ward_q_to_booking_gap",
    "ward_search_to_booking_gap",
    "ward_search_to_completion_gap"
]:
    print(
        f"{col:35s}",
        (ward_intelligence[col] < 0).sum()
    )

print("\nMissing values:")

print(
    ward_intelligence[
        cross_domain_rate_cols
        + [
            "ward_q_to_booking_gap",
            "ward_search_to_booking_gap",
            "ward_search_to_completion_gap"
        ]
    ].isna().sum()
)

END-TO-END FUNNEL VALIDATION

Rate ranges:
ward_search_to_booking_rate         min = 0.0 | max = 0.76
ward_search_to_completion_rate      min = 0.0 | max = 0.5
ward_q_to_booking_rate              min = 0.35 | max = 1.0

Negative end-to-end gaps:
ward_q_to_booking_gap               0
ward_search_to_booking_gap          0
ward_search_to_completion_gap       0

Missing values:
ward_search_to_booking_rate       0
ward_search_to_completion_rate    0
ward_q_to_booking_rate            3
ward_q_to_booking_gap             0
ward_search_to_booking_gap        0
ward_search_to_completion_gap     0
dtype: int64


**Demand pressure vs driver supply**

In [44]:
# ==============================================
# 10.2 Demand pressure vs driver supply
# ==============================================

complete_supply_mask = (
    ward_intelligence["complete_auto_cab_supply"] == 1
)

# Initialise as NaN so incomplete-supply wards remain undefined
pressure_features = [
    "searches_per_active_driver",
    "q_requests_per_active_driver",
    "q_supply_gap_per_active_driver",
    "searches_per_notonride_driver",
    "q_requests_per_notonride_driver",
    "q_supply_gap_per_notonride_driver"
]

for col in pressure_features:
    ward_intelligence[col] = np.nan


# ----------------------------------------------
# Demand relative to total active driver supply
# ----------------------------------------------

ward_intelligence.loc[
    complete_supply_mask,
    "searches_per_active_driver"
] = safe_divide(
    ward_intelligence.loc[
        complete_supply_mask,
        "ward_searches"
    ],
    ward_intelligence.loc[
        complete_supply_mask,
        "active_drvr_auto_cab"
    ]
)

ward_intelligence.loc[
    complete_supply_mask,
    "q_requests_per_active_driver"
] = safe_divide(
    ward_intelligence.loc[
        complete_supply_mask,
        "ward_q_requests"
    ],
    ward_intelligence.loc[
        complete_supply_mask,
        "active_drvr_auto_cab"
    ]
)

ward_intelligence.loc[
    complete_supply_mask,
    "q_supply_gap_per_active_driver"
] = safe_divide(
    ward_intelligence.loc[
        complete_supply_mask,
        "ward_q_supply_gap"
    ],
    ward_intelligence.loc[
        complete_supply_mask,
        "active_drvr_auto_cab"
    ]
)


# ----------------------------------------------
# Demand relative to not-on-ride driver supply
# ----------------------------------------------

ward_intelligence.loc[
    complete_supply_mask,
    "searches_per_notonride_driver"
] = safe_divide(
    ward_intelligence.loc[
        complete_supply_mask,
        "ward_searches"
    ],
    ward_intelligence.loc[
        complete_supply_mask,
        "notonride_drvr_auto_cab"
    ]
)

ward_intelligence.loc[
    complete_supply_mask,
    "q_requests_per_notonride_driver"
] = safe_divide(
    ward_intelligence.loc[
        complete_supply_mask,
        "ward_q_requests"
    ],
    ward_intelligence.loc[
        complete_supply_mask,
        "notonride_drvr_auto_cab"
    ]
)

ward_intelligence.loc[
    complete_supply_mask,
    "q_supply_gap_per_notonride_driver"
] = safe_divide(
    ward_intelligence.loc[
        complete_supply_mask,
        "ward_q_supply_gap"
    ],
    ward_intelligence.loc[
        complete_supply_mask,
        "notonride_drvr_auto_cab"
    ]
)

print("Demand-supply pressure features created successfully.")

Demand-supply pressure features created successfully.


**Validation**

In [45]:
print("DEMAND-SUPPLY PRESSURE VALIDATION")
print("=" * 55)

print("\nMissing values:")

print(
    ward_intelligence[
        pressure_features
    ].isna().sum()
)

print("\nIncomplete-supply wards:")

display(
    ward_intelligence.loc[
        ward_intelligence["complete_auto_cab_supply"] == 0,
        [
            "ward_num",
            "complete_auto_cab_supply"
        ] + pressure_features
    ]
)

print("\nFeature ranges for complete-supply wards:")

for col in pressure_features:
    valid_values = ward_intelligence.loc[
        complete_supply_mask,
        col
    ]

    print(
        f"{col:40s}",
        "min =", round(valid_values.min(), 4),
        "| max =", round(valid_values.max(), 4)
    )

DEMAND-SUPPLY PRESSURE VALIDATION

Missing values:
searches_per_active_driver           2
q_requests_per_active_driver         2
q_supply_gap_per_active_driver       2
searches_per_notonride_driver        2
q_requests_per_notonride_driver      2
q_supply_gap_per_notonride_driver    2
dtype: int64

Incomplete-supply wards:


,ward_num,complete_auto_cab_supply,searches_per_active_driver,q_requests_per_active_driver,q_supply_gap_per_active_driver,searches_per_notonride_driver,q_requests_per_notonride_driver,q_supply_gap_per_notonride_driver
63,b_156,0,NaN,NaN,NaN,NaN,NaN,NaN
218,b_77,0,NaN,NaN,NaN,NaN,NaN,NaN



Feature ranges for complete-supply wards:
searches_per_active_driver               min = 0.0417 | max = 7.0875
q_requests_per_active_driver             min = 0.0 | max = 6.5918
q_supply_gap_per_active_driver           min = 0.0 | max = 5.7755
searches_per_notonride_driver            min = 0.0435 | max = 10.1292
q_requests_per_notonride_driver          min = 0.0 | max = 9.4208
q_supply_gap_per_notonride_driver        min = 0.0 | max = 8.2542


### 10.3 Diagnostic Funnel-Loss and Supply-Pressure Features

Additional diagnostic features are created to separate end-to-end demand
loss into major stages of the mobility funnel.

The features quantify:

- Search-to-Q-stage loss
- Q-stage-to-booking loss
- Booking-to-completion loss
- Overall search-to-completion loss

The quote-stage supply gap is also decomposed using driver utilization.

A high supply-gap rate combined with high driver utilization indicates
potential capacity pressure, while a high supply-gap rate combined with a
large not-on-ride driver share may indicate matching, positioning, or
allocation inefficiency.

These features are diagnostic proxies rather than causal conclusions.

In [46]:
# =====================================================
# 10.3 Funnel-loss diagnostic features
# =====================================================

# Search demand that failed to reach successful Q response
ward_intelligence["pre_q_loss_rate"] = safe_divide(
    ward_intelligence["ward_search_to_q_received_gap"],
    ward_intelligence["ward_searches"]
)

# Successful Q responses that failed to become bookings
ward_intelligence["q_to_booking_loss_rate"] = safe_divide(
    ward_intelligence["ward_q_to_booking_gap"],
    ward_intelligence["ward_q_received"]
)

# Bookings that failed to become completed rides
ward_intelligence["booking_to_completion_loss_rate"] = safe_divide(
    ward_intelligence["ward_booking_to_completion_gap"],
    ward_intelligence["ward_bookings"]
)

# Total searches that failed to become completed rides
ward_intelligence["overall_search_completion_loss_rate"] = safe_divide(
    ward_intelligence["ward_search_to_completion_gap"],
    ward_intelligence["ward_searches"]
)

print("Funnel-loss diagnostic features created.")

Funnel-loss diagnostic features created.


**validation**

In [47]:
loss_rate_cols = [
    "pre_q_loss_rate",
    "q_to_booking_loss_rate",
    "booking_to_completion_loss_rate",
    "overall_search_completion_loss_rate"
]

print("FUNNEL-LOSS VALIDATION")
print("=" * 55)

for col in loss_rate_cols:
    print(
        f"{col:40s}",
        "min =", round(ward_intelligence[col].min(), 4),
        "| max =", round(ward_intelligence[col].max(), 4)
    )

print("\nMissing values:")

print(
    ward_intelligence[
        loss_rate_cols
    ].isna().sum()
)

FUNNEL-LOSS VALIDATION
pre_q_loss_rate                          min = 0.24 | max = 1.0
q_to_booking_loss_rate                   min = 0.0 | max = 0.65
booking_to_completion_loss_rate          min = 0.0 | max = 1.0
overall_search_completion_loss_rate      min = 0.5 | max = 1.0

Missing values:
pre_q_loss_rate                        0
q_to_booking_loss_rate                 3
booking_to_completion_loss_rate        3
overall_search_completion_loss_rate    0
dtype: int64


### 10.4 Supply Shortage vs Allocation Pressure

### Decompose Q-Supply Failure

The ward-level Q-stage supply-gap rate is decomposed using driver
utilization.

Two complementary diagnostic indicators are created:

- `capacity_pressure_score`:
  supply failure occurring while drivers are highly utilized.

- `allocation_pressure_score`:
  supply failure occurring while a larger share of drivers is currently
  not on a ride.

For wards with complete supply data, the two components sum back to the
Q-stage supply-gap rate.

This decomposition helps distinguish potential shortage conditions from
possible matching or driver-positioning inefficiencies.

In [48]:
# =====================================================
# 10.4 Supply pressure decomposition
# =====================================================

diagnostic_supply_mask = (
    (ward_intelligence["complete_auto_cab_supply"] == 1)
    & ward_intelligence["ward_q_supply_gap_rate"].notna()
)

# Initialise as undefined
ward_intelligence["capacity_pressure_score"] = np.nan
ward_intelligence["allocation_pressure_score"] = np.nan

# ---------------------------------------------
# Gap occurring while drivers are highly used
# ---------------------------------------------

ward_intelligence.loc[
    diagnostic_supply_mask,
    "capacity_pressure_score"
] = (
    ward_intelligence.loc[
        diagnostic_supply_mask,
        "ward_q_supply_gap_rate"
    ]
    *
    ward_intelligence.loc[
        diagnostic_supply_mask,
        "ward_driver_utilization_rate"
    ]
)

# ---------------------------------------------
# Gap occurring while drivers are not on rides
# ---------------------------------------------

ward_intelligence.loc[
    diagnostic_supply_mask,
    "allocation_pressure_score"
] = (
    ward_intelligence.loc[
        diagnostic_supply_mask,
        "ward_q_supply_gap_rate"
    ]
    *
    ward_intelligence.loc[
        diagnostic_supply_mask,
        "ward_notonride_driver_share"
    ]
)

print("Supply-pressure decomposition created.")

Supply-pressure decomposition created.


**validation**

In [49]:
print("SUPPLY-PRESSURE DECOMPOSITION VALIDATION")
print("=" * 60)

pressure_score_cols = [
    "capacity_pressure_score",
    "allocation_pressure_score"
]

print("\nScore ranges:")

for col in pressure_score_cols:
    print(
        f"{col:35s}",
        "min =", round(ward_intelligence[col].min(), 4),
        "| max =", round(ward_intelligence[col].max(), 4)
    )

# The two components should reconstruct the Q supply-gap rate
ward_intelligence["pressure_score_sum"] = (
    ward_intelligence["capacity_pressure_score"]
    + ward_intelligence["allocation_pressure_score"]
)

comparison = ward_intelligence.loc[
    diagnostic_supply_mask,
    [
        "pressure_score_sum",
        "ward_q_supply_gap_rate"
    ]
].copy()

comparison["difference"] = (
    comparison["pressure_score_sum"]
    - comparison["ward_q_supply_gap_rate"]
).abs()

print("\nMaximum reconstruction difference:")

print(
    comparison["difference"].max()
)

print("\nMissing values:")

print(
    ward_intelligence[
        pressure_score_cols
    ].isna().sum()
)

SUPPLY-PRESSURE DECOMPOSITION VALIDATION

Score ranges:
capacity_pressure_score             min = 0.0 | max = 0.3372
allocation_pressure_score           min = 0.0 | max = 0.9744

Maximum reconstruction difference:
1.1102230246251565e-16

Missing values:
capacity_pressure_score      3
allocation_pressure_score    3
dtype: int64


**Why this is useful**


Suppose two wards both have:
Q supply gap rate = 50%

But Ward A has:
driver utilization = 90%

Then:
capacity pressure
= 0.50 × 0.90
= 0.45

This suggests most of the supply failure is happening while drivers are already busy.
Ward B instead has:
driver utilization = 20%
not-on-ride share   = 80%

Then:
capacity pressure   = 0.50 × 0.20 = 0.10
allocation pressure = 0.50 × 0.80 = 0.40

That tells NammaFlow something very different:
“Drivers exist, but demand is still not being served.”

That may point toward driver positioning, matching efficiency, dispatch, or geographic imbalance, rather than simply needing more drivers.

In [50]:
print("SUPPLY-PRESSURE DECOMPOSITION VALIDATION")
print("=" * 60)

pressure_score_cols = [
    "capacity_pressure_score",
    "allocation_pressure_score"
]

print("\nScore ranges:")

for col in pressure_score_cols:
    print(
        f"{col:35s}",
        "min =", round(ward_intelligence[col].min(), 4),
        "| max =", round(ward_intelligence[col].max(), 4)
    )

ward_intelligence["pressure_score_sum"] = (
    ward_intelligence["capacity_pressure_score"]
    + ward_intelligence["allocation_pressure_score"]
)

comparison = ward_intelligence.loc[
    diagnostic_supply_mask,
    [
        "pressure_score_sum",
        "ward_q_supply_gap_rate"
    ]
].copy()

comparison["difference"] = (
    comparison["pressure_score_sum"]
    - comparison["ward_q_supply_gap_rate"]
).abs()

print("\nMaximum reconstruction difference:")
print(comparison["difference"].max())

print("\nMissing values:")
print(
    ward_intelligence[
        pressure_score_cols
    ].isna().sum()
)

SUPPLY-PRESSURE DECOMPOSITION VALIDATION

Score ranges:
capacity_pressure_score             min = 0.0 | max = 0.3372
allocation_pressure_score           min = 0.0 | max = 0.9744

Maximum reconstruction difference:
1.1102230246251565e-16

Missing values:
capacity_pressure_score      3
allocation_pressure_score    3
dtype: int64


capacity_pressure_score
= ward_q_supply_gap_rate * ward_driver_utilization_rate


allocation_pressure_score
= ward_q_supply_gap_rate * ward_notonride_driver_share

The idea is:
- Capacity pressure → shortage caused by drivers already being utilized.
- Allocation pressure → drivers exist but are not currently on rides, suggesting matching/allocation inefficiency.

In [51]:
# ============================================================
# INVESTIGATE MISSING SUPPLY-PRESSURE SCORES
# ============================================================

pressure_missing_mask = ward_intelligence[
    ["capacity_pressure_score", "allocation_pressure_score"]
].isna().any(axis=1)

missing_pressure_rows = ward_intelligence.loc[
    pressure_missing_mask
].copy()

print("Rows with missing pressure decomposition:")
print(len(missing_pressure_rows))

cols_to_show = [
    "ward_num",
    "ward_q_supply_gap_rate",
    "ward_driver_utilization_rate",
    "ward_notonride_driver_share",
    "capacity_pressure_score",
    "allocation_pressure_score"
]

cols_to_show = [
    col for col in cols_to_show
    if col in ward_intelligence.columns
]

display(
    missing_pressure_rows[cols_to_show]
)

Rows with missing pressure decomposition:
3


,ward_num,ward_q_supply_gap_rate,ward_driver_utilization_rate,ward_notonride_driver_share,capacity_pressure_score,allocation_pressure_score
63,b_156,0.666667,0.125000,0.875000,NaN,NaN
182,b_44,NaN,0.041667,0.958333,NaN,NaN
218,b_77,0.500000,0.000000,1.000000,NaN,NaN


In [52]:
# ============================================================
# FINAL VALIDATION — STRUCTURAL MISSINGNESS IN PRESSURE SCORES
# ============================================================

affected_wards = ["b_156", "b_44", "b_77"]

print("Affected wards:")
display(
    ward_intelligence.loc[
        ward_intelligence["ward_num"].isin(affected_wards),
        [
            col for col in [
                "ward_num",
                "ward_q_supply_gap_rate",
                "ward_driver_utilization_rate",
                "ward_notonride_driver_share",
                "capacity_pressure_score",
                "allocation_pressure_score"
            ]
            if col in ward_intelligence.columns
        ]
    ]
)

print("\nSupply-related columns available in ward_intelligence:")

supply_related_cols = [
    col for col in ward_intelligence.columns
    if any(
        keyword in col.lower()
        for keyword in ["supply", "driver", "coverage"]
    )
]

print(supply_related_cols)

Affected wards:


,ward_num,ward_q_supply_gap_rate,ward_driver_utilization_rate,ward_notonride_driver_share,capacity_pressure_score,allocation_pressure_score
63,b_156,0.666667,0.125000,0.875000,NaN,NaN
182,b_44,NaN,0.041667,0.958333,NaN,NaN
218,b_77,0.500000,0.000000,1.000000,NaN,NaN



Supply-related columns available in ward_intelligence:
['ward_q_supply_gap', 'ward_q_supply_gap_rate', 'ward_driver_cancellations', 'ward_driver_cancel_share', 'supply_vehicle_coverage', 'has_auto_supply', 'has_cab_supply', 'complete_auto_cab_supply', 'ward_driver_utilization_rate', 'ward_notonride_driver_share', 'ward_log_active_drivers', 'ward_driver_count_gap', 'ward_driver_share_sum', 'searches_per_active_driver', 'q_requests_per_active_driver', 'q_supply_gap_per_active_driver', 'searches_per_notonride_driver', 'q_requests_per_notonride_driver', 'q_supply_gap_per_notonride_driver']


In [53]:
# ============================================================
# VERIFY ROOT CAUSE OF PRESSURE-SCORE MISSINGNESS
# ============================================================

affected_wards = ["b_156", "b_44", "b_77"]

diagnostic_cols = [
    "ward_num",
    "supply_vehicle_coverage",
    "has_auto_supply",
    "has_cab_supply",
    "complete_auto_cab_supply",
    "ward_q_supply_gap",
    "ward_q_supply_gap_rate",
    "ward_driver_utilization_rate",
    "ward_notonride_driver_share",
    "capacity_pressure_score",
    "allocation_pressure_score"
]

diagnostic_cols = [
    col for col in diagnostic_cols
    if col in ward_intelligence.columns
]

display(
    ward_intelligence.loc[
        ward_intelligence["ward_num"].isin(affected_wards),
        diagnostic_cols
    ].sort_values("ward_num")
)

,ward_num,supply_vehicle_coverage,has_auto_supply,has_cab_supply,complete_auto_cab_supply,ward_q_supply_gap,ward_q_supply_gap_rate,ward_driver_utilization_rate,ward_notonride_driver_share,capacity_pressure_score,allocation_pressure_score
63,b_156,1,1,0,0,2,0.666667,0.125000,0.875000,NaN,NaN
182,b_44,2,1,1,1,0,NaN,0.041667,0.958333,NaN,NaN
218,b_77,1,1,0,0,4,0.500000,0.000000,1.000000,NaN,NaN


### 10.4 Supply-Pressure Decomposition — Validation

The ward-level Q-stage supply-gap pressure was decomposed into two operational components:

- **Capacity Pressure Score** — represents supply pressure associated with drivers already being actively utilized.
- **Allocation Pressure Score** — represents supply pressure associated with drivers who are available but not currently on rides.

Validation confirmed that:

\[
\text{Capacity Pressure} + \text{Allocation Pressure}
\approx \text{Q Supply Gap Rate}
\]

with a maximum reconstruction difference of approximately `1.11e-16`, which is negligible floating-point error.

Three wards contain missing decomposition scores:

- `b_156` — incomplete Auto + Cab driver-supply coverage.
- `b_77` — incomplete Auto + Cab driver-supply coverage.
- `b_44` — complete supply coverage, but the Q-stage supply-gap rate is undefined because the observed Q supply gap is zero relative to its calculation denominator.

These missing values are intentionally preserved rather than imputed with zero, because zero would represent confirmed absence of pressure, whereas these cases represent incomplete or undefined operational measurements.

This decomposition allows NammaFlow to distinguish between:

1. **Capacity shortages** — insufficient effective driver capacity.
2. **Allocation inefficiencies** — available supply exists but is not effectively converted into active ride fulfillment.

## 11. Final Feature Audit and Validation

Before exporting the Phase 3 engineered datasets, a final audit is performed to verify:

- dataset dimensions,
- ward uniqueness,
- duplicate records,
- missing-value patterns,
- infinite values,
- feature ranges,
- supply-coverage quality,
- and the integrity of the newly engineered cross-domain features.

Structural missingness identified during feature engineering is preserved where appropriate rather than incorrectly imputed.

In [55]:
# ============================================================
# FINAL PHASE 3 FEATURE AUDIT
# ============================================================

import numpy as np
import pandas as pd

print("=" * 65)
print("NAMMAFLOW — FINAL PHASE 3 FEATURE AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 1. DATASET SHAPE
# ------------------------------------------------------------

print("\n1. DATASET SHAPE")
print("-" * 65)

print(f"ward_intelligence shape : {ward_intelligence.shape}")
print(f"Total wards             : {ward_intelligence['ward_num'].nunique()}")
print(f"Total columns           : {ward_intelligence.shape[1]}")


# ------------------------------------------------------------
# 2. WARD UNIQUENESS
# ------------------------------------------------------------

print("\n2. WARD UNIQUENESS")
print("-" * 65)

duplicate_wards = ward_intelligence["ward_num"].duplicated().sum()

print(f"Duplicate ward rows : {duplicate_wards}")

if duplicate_wards == 0:
    print("PASS: One row per ward.")
else:
    print("WARNING: Duplicate ward records detected.")


# ------------------------------------------------------------
# 3. EXACT DUPLICATES
# ------------------------------------------------------------

print("\n3. EXACT DUPLICATE ROWS")
print("-" * 65)

exact_duplicates = ward_intelligence.duplicated().sum()

print(f"Exact duplicate rows : {exact_duplicates}")


# ------------------------------------------------------------
# 4. MISSING VALUES
# ------------------------------------------------------------

print("\n4. MISSING VALUES")
print("-" * 65)

missing_summary = (
    ward_intelligence
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = missing_summary[missing_summary > 0]

print(f"Columns containing missing values : {len(missing_summary)}")

display(
    missing_summary
    .rename("missing_count")
    .to_frame()
)


# ------------------------------------------------------------
# 5. INFINITE VALUES
# ------------------------------------------------------------

print("\n5. INFINITE VALUES")
print("-" * 65)

numeric_cols = ward_intelligence.select_dtypes(
    include=np.number
).columns

inf_counts = pd.Series({
    col: np.isinf(ward_intelligence[col]).sum()
    for col in numeric_cols
})

inf_counts = inf_counts[inf_counts > 0]

print(f"Columns containing infinite values : {len(inf_counts)}")

if len(inf_counts) == 0:
    print("PASS: No infinite numeric values detected.")
else:
    display(inf_counts.to_frame("infinite_count"))


# ------------------------------------------------------------
# 6. SUPPLY COVERAGE
# ------------------------------------------------------------

print("\n6. SUPPLY COVERAGE")
print("-" * 65)

if "complete_auto_cab_supply" in ward_intelligence.columns:

    supply_coverage_summary = (
        ward_intelligence["complete_auto_cab_supply"]
        .value_counts(dropna=False)
        .sort_index()
    )

    display(
        supply_coverage_summary
        .rename("ward_count")
        .to_frame()
    )


# ------------------------------------------------------------
# 7. CROSS-DOMAIN FEATURE CHECK
# ------------------------------------------------------------

print("\n7. CROSS-DOMAIN FEATURES")
print("-" * 65)

cross_domain_features = [
    "ward_q_supply_gap",
    "ward_q_supply_gap_rate",
    "searches_per_active_driver",
    "q_requests_per_active_driver",
    "q_supply_gap_per_active_driver",
    "searches_per_notonride_driver",
    "q_requests_per_notonride_driver",
    "q_supply_gap_per_notonride_driver",
    "capacity_pressure_score",
    "allocation_pressure_score"
]

available_cross_domain_features = [
    col for col in cross_domain_features
    if col in ward_intelligence.columns
]

print("Available cross-domain features:")
for col in available_cross_domain_features:
    print(f" - {col}")


# ------------------------------------------------------------
# 8. NUMERIC RANGE CHECK
# ------------------------------------------------------------

print("\n8. FEATURE RANGE CHECK")
print("-" * 65)

range_summary = pd.DataFrame({
    "min": ward_intelligence[
        available_cross_domain_features
    ].min(),

    "max": ward_intelligence[
        available_cross_domain_features
    ].max(),

    "missing": ward_intelligence[
        available_cross_domain_features
    ].isna().sum()
})

display(range_summary)


print("\n" + "=" * 65)
print("FINAL AUDIT COMPLETE")
print("=" * 65)

NAMMAFLOW — FINAL PHASE 3 FEATURE AUDIT

1. DATASET SHAPE
-----------------------------------------------------------------
ward_intelligence shape : (244, 65)
Total wards             : 244
Total columns           : 65

2. WARD UNIQUENESS
-----------------------------------------------------------------
Duplicate ward rows : 0
PASS: One row per ward.

3. EXACT DUPLICATE ROWS
-----------------------------------------------------------------
Exact duplicate rows : 0

4. MISSING VALUES
-----------------------------------------------------------------
Columns containing missing values : 23


,missing_count
ward_other_cancel_share,19
ward_rider_cancel_share,19
ward_driver_cancel_share,19
ward_earning_per_completed_ride,7
ward_earning_per_ride,3
ward_ride_completion_rate,3
ward_booking_completion_rate,3
ward_cancel_per_booking,3
ward_booking_to_ride_rate,3
booking_to_completion_loss_rate,3



5. INFINITE VALUES
-----------------------------------------------------------------
Columns containing infinite values : 0
PASS: No infinite numeric values detected.

6. SUPPLY COVERAGE
-----------------------------------------------------------------


,ward_count
complete_auto_cab_supply,
0,2
1,242



7. CROSS-DOMAIN FEATURES
-----------------------------------------------------------------
Available cross-domain features:
 - ward_q_supply_gap
 - ward_q_supply_gap_rate
 - searches_per_active_driver
 - q_requests_per_active_driver
 - q_supply_gap_per_active_driver
 - searches_per_notonride_driver
 - q_requests_per_notonride_driver
 - q_supply_gap_per_notonride_driver
 - capacity_pressure_score
 - allocation_pressure_score

8. FEATURE RANGE CHECK
-----------------------------------------------------------------


,min,max,missing
ward_q_supply_gap,0.000000,3962.000000,0
ward_q_supply_gap_rate,0.000000,1.000000,1
searches_per_active_driver,0.041667,7.087464,2
q_requests_per_active_driver,0.000000,6.591837,2
q_supply_gap_per_active_driver,0.000000,5.775510,2
searches_per_notonride_driver,0.043478,10.129167,2
q_requests_per_notonride_driver,0.000000,9.420833,2
q_supply_gap_per_notonride_driver,0.000000,8.254167,2
capacity_pressure_score,0.000000,0.337237,3
allocation_pressure_score,0.000000,0.974359,3



FINAL AUDIT COMPLETE


### 11.1 Final Audit Summary

The final Phase 3 feature audit confirmed that the ward-level intelligence dataset is structurally valid and ready for export.

**Dataset integrity**

- Final ward-level shape: **244 rows × 65 columns**
- Unique wards: **244**
- Duplicate ward records: **0**
- Exact duplicate rows: **0**
- Infinite numeric values: **0**

**Driver-supply coverage**

- **242 wards** contain complete Auto + Cab supply coverage.
- **2 wards (`b_156`, `b_77`)** contain incomplete supply coverage and are explicitly identified through the supply-quality features.

**Missing-value handling**

A total of 23 engineered columns contain at least one missing value. These missing values are primarily structural and occur where a ratio or operational metric is mathematically undefined, such as zero-denominator cases, or where driver-supply coverage is incomplete.

No blanket imputation is applied because replacing these values with zero would incorrectly convert "undefined / unavailable" measurements into confirmed zero operational activity.

**Cross-domain feature validation**

Demand, funnel, driver-supply, utilization, and pressure features were successfully integrated into the ward-level intelligence dataset.

The final feature ranges were checked and no invalid infinite values or obvious range violations were detected.

The engineered dataset is therefore considered ready for downstream modelling, scoring, segmentation, and operational recommendation development.

In [56]:
# ============================================================
# 12. EXPORT FEATURE-ENGINEERED WARD INTELLIGENCE DATASET
# ============================================================

from pathlib import Path

feature_dir = Path("data/features")
feature_dir.mkdir(parents=True, exist_ok=True)

ward_intelligence_path = feature_dir / "ward_intelligence.csv"

ward_intelligence.to_csv(
    ward_intelligence_path,
    index=False
)

print("=" * 65)
print("WARD INTELLIGENCE EXPORT")
print("=" * 65)

print(f"\nSaved to : {ward_intelligence_path}")
print(f"Shape    : {ward_intelligence.shape}")
print(f"Rows     : {len(ward_intelligence):,}")
print(f"Columns  : {ward_intelligence.shape[1]}")

print("\nFile exists :", ward_intelligence_path.exists())

if ward_intelligence_path.exists():
    print(
        "File size   :",
        f"{ward_intelligence_path.stat().st_size / 1024:.2f} KB"
    )

WARD INTELLIGENCE EXPORT

Saved to : data/features/ward_intelligence.csv
Shape    : (244, 65)
Rows     : 244
Columns  : 65

File exists : True
File size   : 139.99 KB


In [57]:
# ============================================================
# 12.1 VERIFY EXPORTED WARD INTELLIGENCE DATASET
# ============================================================

ward_intelligence_check = pd.read_csv(
    ward_intelligence_path
)

print("=" * 65)
print("WARD INTELLIGENCE — EXPORT VERIFICATION")
print("=" * 65)

print(f"\nReloaded shape       : {ward_intelligence_check.shape}")
print(
    f"Unique wards         : "
    f"{ward_intelligence_check['ward_num'].nunique()}"
)
print(
    f"Duplicate ward rows  : "
    f"{ward_intelligence_check['ward_num'].duplicated().sum()}"
)
print(
    f"Exact duplicate rows : "
    f"{ward_intelligence_check.duplicated().sum()}"
)

same_columns = (
    list(ward_intelligence_check.columns)
    == list(ward_intelligence.columns)
)

print(f"Column order matches  : {same_columns}")

expected_shape = (244, 65)

if (
    ward_intelligence_check.shape == expected_shape
    and ward_intelligence_check["ward_num"].nunique() == 244
    and ward_intelligence_check["ward_num"].duplicated().sum() == 0
    and ward_intelligence_check.duplicated().sum() == 0
    and same_columns
):
    print("\nPASS: Exported ward intelligence dataset verified successfully.")
else:
    print("\nWARNING: Export verification requires investigation.")

WARD INTELLIGENCE — EXPORT VERIFICATION

Reloaded shape       : (244, 65)
Unique wards         : 244
Duplicate ward rows  : 0
Exact duplicate rows : 0
Column order matches  : True

PASS: Exported ward intelligence dataset verified successfully.


## 13. Phase 3 — Feature Engineering Summary

Phase 3 transformed the cleaned NammaFlow datasets into modelling- and decision-ready features across temporal, demand, funnel, operational, driver-supply, and ward-level domains.

### Key feature-engineering work completed

**Temporal Features**
- Day-of-week and day-name features
- Time-period segmentation
- Cyclical hour encoding using sine and cosine transformations

**Demand & Funnel Features**
- Search-to-E stage progression
- E-to-Q progression
- Q-stage acceptance and supply-gap metrics
- Booking conversion and funnel-loss metrics
- Booking-to-ride and ride-completion behaviour

**Ward-Level Operational Features**
- Ward search and booking intensity
- Ride completion metrics
- Cancellation behaviour
- Earnings and trip-efficiency metrics

**Driver-Supply Features**
- Auto and Cab supply coverage
- Active-driver availability
- Driver utilization
- Not-on-ride driver share
- Supply-quality flags

**Cross-Domain Demand–Supply Features**
- Searches per active driver
- Q requests per active driver
- Q supply gap per active driver
- Searches per not-on-ride driver
- Q requests per not-on-ride driver
- Q supply gap per not-on-ride driver

**Supply-Pressure Decomposition**
- Capacity Pressure Score
- Allocation Pressure Score

The pressure decomposition separates demand–supply imbalance into:

1. **Capacity pressure** — pressure associated with already-utilized driver capacity.
2. **Allocation pressure** — pressure associated with available but underutilized driver supply.

### Final Ward Intelligence Dataset

The final engineered ward-level dataset contains:

- **244 wards**
- **65 features**
- **0 duplicate ward records**
- **0 exact duplicate rows**
- **0 infinite numeric values**
- **242 wards with complete Auto + Cab supply coverage**

Structural missing values are intentionally retained where metrics are mathematically undefined or where driver-supply coverage is incomplete.

The final dataset was exported as:

`data/features/ward_intelligence.csv`

and successfully reloaded and validated with the expected shape of **(244, 65)**.

### Phase 3 Status

**Phase 3 — Feature Engineering: COMPLETE**

The project is now ready to proceed to the next analytical phase using the engineered NammaFlow feature set.

In [59]:
# ============================================================
# GIT STATUS BEFORE PHASE 3 COMMIT
# ============================================================

!pwd
!git rev-parse --show-toplevel
!git branch --show-current
!git status --short
!git log -1 --oneline

/content/NammaFlow
/content/NammaFlow
main
?? data/features/
c996b71 (HEAD -> main, origin/main, origin/HEAD) Remove duplicate Phase 2 notebook from repository root


In [60]:
# ============================================================
# CHECK PHASE 3 NOTEBOOK LOCATION
# ============================================================

print("Repository notebooks:")
!ls -lh notebooks/

print("\nGit status for notebooks:")
!git status --short notebooks/

print("\nSearching for Phase 3 notebook in Colab/Drive:")
!find /content -name "03_feature_engineering.ipynb" -type f 2>/dev/null

Repository notebooks:
total 2.0M
-rw-r--r-- 1 root root 338K Oct  7 17:53 01_data_understanding.ipynb
-rw-r--r-- 1 root root 1.7M Oct  7 17:53 02_exploratory_data_analysis.ipynb

Git status for notebooks:

Searching for Phase 3 notebook in Colab/Drive:


In [61]:
# ============================================================
# LOCATE PHASE 3 NOTEBOOK IN GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

print("\nSearching Google Drive for Phase 3 notebook...\n")

!find /content/drive/MyDrive \
    -name "03_feature_engineering.ipynb" \
    -type f 2>/dev/null

Mounted at /content/drive

Searching Google Drive for Phase 3 notebook...

/content/drive/MyDrive/Colab Notebooks/03_feature_engineering.ipynb


In [62]:
# ============================================================
# COPY COMPLETED PHASE 3 NOTEBOOK INTO GIT REPOSITORY
# ============================================================

source_notebook = "/content/drive/MyDrive/Colab Notebooks/03_feature_engineering.ipynb"
repo_notebook = "/content/NammaFlow/notebooks/03_feature_engineering.ipynb"

!cp "$source_notebook" "$repo_notebook"

print("Notebook copied into repository.\n")

print("Repository notebook:")
!ls -lh "$repo_notebook"

print("\nVerify source and repository copy:")
!sha256sum "$source_notebook"
!sha256sum "$repo_notebook"

print("\nGit status:")
%cd /content/NammaFlow
!git status --short

Notebook copied into repository.

Repository notebook:
-rw------- 1 root root 368K Oct  7 18:19 /content/NammaFlow/notebooks/03_feature_engineering.ipynb

Verify source and repository copy:
56f6834cb87a93726a05343bf51d44c6e904430aacb769f30e7d4665f07c59e5  /content/drive/MyDrive/Colab Notebooks/03_feature_engineering.ipynb
56f6834cb87a93726a05343bf51d44c6e904430aacb769f30e7d4665f07c59e5  /content/NammaFlow/notebooks/03_feature_engineering.ipynb

Git status:
/content/NammaFlow
?? data/features/
?? notebooks/03_feature_engineering.ipynb
